# 🔤 26 · Attention, transformers and self-supervised learning
### What protein language models learn: attention from scratch, a tiny transformer pre-trained by masked-residue prediction on a simulated protein family, and what it discovers about 3-D contacts and mutation effects

**BiologyToML** · Level 2 · Research frontier · Part F — Modern machine-learning building blocks · Module 24 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/24_attention_transformers.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1–2 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, PyTorch (CPU), SciPy, scikit-learn, pandas, Matplotlib |
| **Prerequisites** | Level 1 — essential: [Module 22](22_neural_networks.ipynb) (PyTorch, training loops, cross-entropy); helpful: Modules 08 (k-NN), 19–20 (sequence models, profile HMMs) and 23 (honest evaluation) |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **Builds on** | [08 · k-NN and naïve Bayes](08_knn_naive_bayes.ipynb), [19 · HMM inference](19_hmm_inference.ipynb), [20 · HMM learning](20_hmm_learning.ipynb), [22 · Neural networks](22_neural_networks.ipynb), [23 · Model selection](23_model_selection.ipynb) |
| **Research frontier covered** | transformers, masked-language-model pre-training, protein and genomic language models, single-cell foundation models — see [`FRONTIER_MAP.md`](../FRONTIER_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All sequences are **SIMULATED**. The 3-D structure is an idealised four-helix bundle (`simulate_helix_bundle`,
> 68 residues, known contact map). The protein family (3,000 aligned sequences) is sampled in Section 1 from a Potts
> model whose couplings join residues in contact and follow three simple chemical rules (hydrophobic packing, salt
> bridges, size complementarity). The hidden truth is therefore known exactly: the contact map, the fields and couplings,
> and the effect of every single mutation (`deep_mutational_scan`). Real protein language models are trained on
> hundreds of millions of unaligned natural sequences from all families; our toy trains in about a minute on a CPU,
> so that every claim can be checked against the truth.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** why the best residue at one position of a protein depends on its context (its 3-D neighbours), and how tokens become learned embedding vectors.
2. **Compute** scaled dot-product attention by hand and in NumPy, **justify** the $1/\sqrt{d_k}$ factor, and **interpret** attention as a learned, soft k-nearest-neighbour average.
3. **Verify** numerically that self-attention is permutation-equivariant and **explain** how sinusoidal or learned positional encodings give it a sense of order.
4. **Build** a transformer encoder (multi-head attention, residual connections, LayerNorm, MLP) in PyTorch, **check** it against PyTorch's built-in attention and **count** its parameters.
5. **Pre-train** it by masked language modelling and **evaluate** it with masked-residue accuracy and perplexity against a uniform model, an independent-site model and the true (oracle) model.
6. **Probe** what self-supervision learned: contacts from attention maps and from a categorical Jacobian, zero-shot variant effects against a deep mutational scan, and a linear probe for secondary structure — each against ground truth.
7. **Write** the InfoNCE contrastive loss and **contrast** contrastive with masked pre-training.
8. **Discuss** scaling, tokenisation, the $O(L^2)$ cost of attention, probing vs fine-tuning vs zero-shot use, and how protein, genomic and single-cell foundation models are evaluated.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, PyTorch, course toolkit | 5 min |
| 1 · Tokens, embeddings and context | a simulated protein family; conservation and co-variation; embeddings | 20 min |
| 2 · Attention from scratch | Q, K, V; √d; attention as soft k-NN; equivariance; positional encodings | 25 min |
| 3 · The transformer block | multi-head attention, residuals, LayerNorm, MLP; check vs PyTorch; parameters | 20 min |
| 4 · Masked language modelling | BERT masking, training, 🧪 perplexity vs independent-site and oracle | 25 min |
| 5 · 🧪 What did it learn? Structure | attention maps and categorical Jacobian vs true contacts | 20 min |
| 6 · 🧪 What did it learn? Function | zero-shot variant effects vs a deep mutational scan; linear probe | 20 min |
| 7 · Contrastive learning in one page | InfoNCE on two maskings of the same sequence | 10 min |
| 8 · Scaling and foundation models | O(L²) memory, tokenisation (k-mers, BPE), how models are used | 15 min |
| 9 · 🧭 From toy to real data | UniRef, ESM-2, DNA and single-cell models, evaluation caveats | 10 min |
| ✅ / 🧠 / 🚀 / 📝 | takeaways, checkpoint, discussion, going further, lab note | 10 min |

## 🧫 The biological question

UniProt holds hundreds of millions of protein sequences, and for the vast majority nobody has ever measured a
structure or a function. Yet each of these sequences is the survivor of an enormous experiment: evolution has tried
mutations at every position and kept only the variants that still fold and work. Two traces of this experiment are
written into the sequences of a protein family. **Conservation**: positions that matter tolerate few amino acids.
**Co-variation**: two residues that touch in the folded protein must stay compatible, so when one changes (a small
residue becomes a large one), its partner tends to change too (a large residue becomes a small one).

In 2019–2021 a surprising result emerged. Large neural networks called **transformers**, trained on nothing but raw
sequences with a fill-in-the-blank game — hide 15 % of the residues, predict them from the rest — learned internal
representations that encode 3-D contacts (Rives *et al.* 2021, who trained on databases of tens to hundreds of
millions of sequences; Rao
*et al.* 2021 showed that attention maps track contacts) and that score the effect of mutations without ever seeing a
measured phenotype (Meier *et al.* 2021, ESM-1v). By 2023 the ESM-2 family (Lin *et al.* 2023) was accurate enough to
drive a structure predictor, ESMFold. The same recipe has since been applied to DNA and to single-cell transcriptomes.

This notebook rebuilds the whole idea at toy scale, with **known ground truth** at every step:

* a SIMULATED family of 3,000 sequences of a 68-residue four-helix bundle, sampled from a model in which residues in
  contact are coupled — a miniature of the multiple sequence alignments in Pfam;
* attention and a transformer written from scratch, pre-trained by masked-residue prediction in about a minute;
* checks of what it learned against the true contact map and against a simulated **deep mutational scan** (DMS) of
  every single mutant — the kind of experiment collected in the ProteinGym benchmark.

**Can a model trained only to fill in masked residues discover which residues touch and which mutations are harmful?**

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this notebook uses PyTorch (preinstalled on Colab); a CPU is enough.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import math
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.special import logsumexp as lse
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

In [ ]:
# ── 0.1 PyTorch configuration ──
torch.manual_seed(SEED)                  # PyTorch has its own random-number generator
torch.set_num_threads(2)                 # small models: 2 CPU threads are plenty (and polite on shared machines)
OKABE_ITO = ["#0072B2", "#D55E00", "#009E73", "#E69F00", "#CC79A7", "#56B4E9"]   # colour-blind-safe palette
print("PyTorch", torch.__version__, "| threads:", torch.get_num_threads())

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`simulate_helix_bundle` builds the 3-D structure and its contact map, `potts_score` computes the statistical fitness of sequences under a Potts model, and `deep_mutational_scan` simulates a measurement of every single mutant of a wild-type sequence. `simulate_protein_family` (random coupling tables, used in Module 37) is included for the comparison suggested in Section 5 and the Bonus. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np
import pandas as pd

AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"


def simulate_helix_bundle(n_helices: int = 4, helix_length: int = 14, loop_length: int = 4,
                          rng=None, noise: float = 0.0) -> dict:
    """Cα coordinates (Å) of an idealised four-helix-bundle protein, its contact map and secondary structure.

    Helices are ideal right-handed α-helices (3.6 residues per turn, 1.5 Å rise, 2.3 Å radius)
    packed antiparallel on the corners of a 10 Å square and joined by short loops.  ``noise`` adds
    Gaussian jitter (needs ``rng``).  Contacts: Cα–Cα distance < 8 Å and |i − j| ≥ 3.

    Returns a dict with ``coords`` (L × 3), ``ss`` (string of 'H'/'L'), ``distances`` and ``contacts``.
    """
    corners = 5.0 * np.array([[1, 1], [1, -1], [-1, -1], [-1, 1]], float)
    pts, ss = [], ""
    for h in range(n_helices):
        cx, cy = corners[h % 4] * (1 + h // 4)
        up = h % 2 == 0
        k = np.arange(helix_length)
        angle = np.deg2rad(100.0) * k * (1 if up else -1)      # turn with the direction of travel, so
        z = 1.5 * (k - (helix_length - 1) / 2) * (1 if up else -1)   # every helix is right-handed
        helix = np.c_[cx + 2.3 * np.cos(angle), cy + 2.3 * np.sin(angle), z]
        if pts:                                                      # loop: circular arc, 3.8 Å steps
            start, end = pts[-1], helix[0]
            d, m = np.linalg.norm(end - start), loop_length + 1
            lo, hi = 1e-6, 2 * np.pi - 1e-6                              # arc angle Θ with m equal chords
            for _ in range(60):
                theta = (lo + hi) / 2
                lo, hi = (theta, hi) if 3.8 * np.sin(theta / 2) / np.sin(theta / (2 * m)) > d else (lo, theta)
            R = 1.9 / np.sin(theta / (2 * m))
            e1 = (end - start) / d
            e2 = np.r_[0, 0, np.sign(start[2])] - np.dot(np.r_[0, 0, np.sign(start[2])], e1) * e1
            e2 /= np.linalg.norm(e2)
            c = (start + end) / 2 - R * np.cos(theta / 2) * e2
            for k in range(1, m):
                psi = -theta / 2 + k * theta / m
                pts.append(c + R * (np.sin(psi) * e1 + np.cos(psi) * e2))
            ss += "L" * loop_length
        pts.extend(helix)
        ss += "H" * helix_length
    coords = np.array(pts)
    if noise > 0:
        coords = coords + rng.normal(0, noise, coords.shape)
    D = np.linalg.norm(coords[:, None] - coords[None], axis=-1)
    sep = np.abs(np.arange(len(coords))[:, None] - np.arange(len(coords))[None])
    return dict(coords=coords, ss=ss, distances=D, contacts=(D < 8.0) & (sep >= 3))


def potts_score(seqs, h, J) -> np.ndarray:
    """Statistical fitness Σ_i h_i(a_i) + Σ_{i<j} J_ij(a_i, a_j) of integer-coded sequences (higher = fitter)."""
    seqs = np.atleast_2d(seqs)
    L = seqs.shape[1]
    ii, jj = np.nonzero(np.triu(np.abs(J).sum(axis=(2, 3)) > 0, 1))      # coupled pairs only
    return h[np.arange(L), seqs].sum(axis=1) + J[ii, jj, seqs[:, ii], seqs[:, jj]].sum(axis=1)


def simulate_protein_family(rng: np.random.Generator, contacts: np.ndarray, n_seqs: int = 3000,
                            q: int = 20, field_scale: float = 1.0, coupling_scale: float = 1.0,
                            n_sweeps: int = 100) -> dict:
    """Simulate a protein family (a multiple sequence alignment) from a Potts model on a contact map.

    Each residue pair in contact gets a q × q coupling matrix J_ij (pairs that fit together
    are rewarded), each position a field h_i (its conservation).  Sequences are drawn by
    Gibbs sampling (``n_seqs`` independent chains, ``n_sweeps`` sweeps each), so conserved
    positions and *co-varying* contact pairs appear exactly as in natural alignments — but
    without phylogeny.  The wild type is the fittest sampled sequence.

    Returns a dict with ``msa`` (n_seqs × L integers 0…q−1; letters = AMINO_ACIDS[:q]),
    ``h`` (L × q), ``J`` (L × L × q × q, symmetric: J[j, i] = J[i, j].T), ``wild_type``.
    """
    L = contacts.shape[0]
    cons = rng.uniform(0.3, 1.5, L)[:, None] * field_scale
    h = rng.normal(0, 1, (L, q)) * cons
    J = np.zeros((L, L, q, q))
    pairs = [(i, j) for i in range(L) for j in range(i + 1, L) if contacts[i, j]]
    for i, j in pairs:
        J[i, j] = rng.normal(0, 0.5 * coupling_scale, (q, q))
        J[j, i] = J[i, j].T
    nbrs = [np.flatnonzero(contacts[i]) for i in range(L)]
    x = rng.integers(0, q, (n_seqs, L))
    rows = np.arange(n_seqs)
    for _ in range(n_sweeps):
        for i in rng.permutation(L):
            logits = np.tile(h[i], (n_seqs, 1))
            for j in nbrs[i]:
                logits += J[j, i][x[:, j]]                                 # = J_ij(·, x_j)
            g = logits - np.log(-np.log(rng.random((n_seqs, q))))           # Gumbel-max sampling
            x[rows, i] = np.argmax(g, axis=1)
    wild_type = x[np.argmax(potts_score(x, h, J))].copy()
    return dict(msa=x, h=h, J=J, wild_type=wild_type)


def deep_mutational_scan(rng: np.random.Generator, wild_type, h, J, noise: float = 0.15) -> pd.DataFrame:
    """Simulate a deep mutational scan: every single mutant of ``wild_type`` with a measured fitness.

    True effect ΔE = score(mutant) − score(wild type) under the Potts model; the assay reads it
    through a sigmoid ("fraction folded", global epistasis) plus measurement noise.
    Returns a DataFrame with position, wild-type and mutant letters, ``delta_energy`` and ``fitness``.
    """
    wt = np.asarray(wild_type)
    L, q = h.shape
    # ΔE for every substitution i: wt_i → a, computed in one go (only position i changes)
    pair = J[np.arange(L)[:, None], np.arange(L)[None, :], :, wt[None, :]].sum(axis=1)    # L × q
    delta = (h - h[np.arange(L), wt][:, None]) + (pair - pair[np.arange(L), wt][:, None])
    rows = [(i, AMINO_ACIDS[wt[i]], AMINO_ACIDS[a], delta[i, a]) for i in range(L) for a in range(q) if a != wt[i]]
    df = pd.DataFrame(rows, columns=["position", "wt", "mut", "delta_energy"])
    df["fitness"] = 1 / (1 + np.exp(-(df["delta_energy"] + 3.0) / 1.5)) + rng.normal(0, noise, len(df))
    df["mutant"] = df["wt"] + df["position"].astype(str) + df["mut"]
    return df


print("🧰 Toolkit loaded:", ", ".join(['simulate_helix_bundle', 'potts_score', 'simulate_protein_family', 'deep_mutational_scan']))

---
## 1 · Tokens, embeddings and why context matters

> 📍 **Where we are.** In [Module 22](22_neural_networks.ipynb) a 1-D CNN read one-hot DNA and found a motif: a
> *local* pattern. Proteins add a twist: residues far apart in the sequence can touch in 3-D. Before building a model
> we create the data — a protein family with a known structure — and look at the two signals evolution leaves in it.

🔬 **Assay box — multiple sequence alignments (MSAs).** *What is measured:* nothing directly; an MSA is computed by
searching sequence databases (UniProt, metagenomes) for homologues of a protein with tools such as jackhmmer or
HHblits and aligning them so that equivalent residues sit in the same column. *Typical sizes:* tens to hundreds of
thousands of sequences for well-studied families, a handful for "orphan" proteins. *Typical artefacts:* gaps
(insertions and deletions), alignment errors, and **phylogenetic redundancy** — many near-identical sequences from
closely related species, which must be down-weighted ([Module 37](37_protein_sequences_language_models.ipynb) shows how).
Our simulation has no gaps and no phylogeny: every sequence is an independent draw.

**The structure.** `simulate_helix_bundle` builds a 68-residue four-helix bundle: four ideal α-helices of 14 residues
joined by 4-residue loops. Two residues are **in contact** when their Cα atoms are closer than 8 Å (and at least 3
apart in sequence).

In [ ]:
# ── 1.1 The structure: an idealised four-helix bundle (SIMULATED) ──
structure = simulate_helix_bundle()
contacts = structure["contacts"]                                # L × L booleans: Cα–Cα < 8 Å and |i − j| ≥ 3
L = contacts.shape[0]
sep = np.abs(np.arange(L)[:, None] - np.arange(L)[None, :])    # sequence separation |i − j|
long_range = contacts & (sep >= 6)                              # contacts that are not neighbours along one helix
ss = structure["ss"]                                            # 'H' = helix, 'L' = loop
print("SIMULATED protein with L =", L, "residues")
print("secondary structure:", ss)
print(f"contacts: {contacts.sum() // 2} pairs, of which {long_range.sum() // 2} have |i − j| ≥ 6 ('long-range' here)")
n_pairs = len(np.triu_indices(L, 6)[0])
print(f"a random pair with |i − j| ≥ 6 is a contact with probability {long_range.sum() / 2 / n_pairs:.3f}")

**The family.** Natural protein families are often modelled by a **Potts model** (the subject of
[Module 37](37_protein_sequences_language_models.ipynb)): the probability of a sequence $x=(x_1,\dots,x_L)$ is

$$P(x)=\frac{1}{Z}\exp\Big(\sum_{i} h_i(x_i)+\sum_{i<j} J_{ij}(x_i,x_j)\Big).$$

*In words:* every position $i$ has a preference $h_i(a)$ for each amino acid $a$ (conservation), every pair of
positions in contact has a $20\times20$ table $J_{ij}(a,b)$ saying how well residues $a$ and $b$ fit together
(co-variation), and $Z$ makes the probabilities sum to one. It is a Markov random field
([Module 18](18_gaussian_graphical_models.ipynb)) with categorical instead of Gaussian variables; the graph is the
contact map.

What should the tables $J_{ij}$ look like? In real proteins, which residue pairs fit together is not arbitrary: it is
dominated by a few physico-chemical factors (statistical contact potentials derived from structures, such as that of
Miyazawa & Jernigan 1985, are dominated by hydrophobicity). We therefore build every $J_{ij}$ from three shared
"chemistry" matrices — **hydrophobic packing** (oily with oily, polar with polar), **salt bridges** (opposite charges
attract) and **size complementarity** (a big side chain packs against a small one) — mixed in proportions that differ
from contact to contact, plus a little idiosyncratic noise.

In [ ]:
# ── 1.2 Three chemical rules for residues in contact ──
# Values in the order of AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
HYDROPATHY = np.array([1.8, 2.5, -3.5, -3.5, 2.8, -0.4, -3.2, 4.5, -3.9, 3.8,
                       1.9, -3.5, -1.6, -3.5, -4.5, -0.8, -0.7, 4.2, -0.9, -1.3])   # Kyte–Doolittle; + = oily
CHARGE = np.array([0, 0, -1, -1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], float)   # D, E −1; K, R +1
VOLUME = np.array([88.6, 108.5, 111.1, 138.4, 189.9, 60.1, 153.2, 166.7, 168.6, 166.7,
                   162.9, 114.1, 112.7, 143.8, 173.4, 89.0, 116.1, 140.0, 227.8, 193.6])   # residue volume, Å³ (approx.)


def zscore(v):
    """Centre a vector and divide by its standard deviation."""
    return (v - v.mean()) / v.std()


def chemistry_matrices():
    """Three 20 × 20 residue–residue preference matrices, centred (no row/column offsets), largest entry = ±1."""
    raw = {"hydrophobic packing": np.outer(zscore(HYDROPATHY), zscore(HYDROPATHY)),   # like with like
           "salt bridge": -np.outer(CHARGE, CHARGE),                                   # opposite charges attract
           "size complementarity": -np.outer(zscore(VOLUME), zscore(VOLUME))}         # big packs against small
    out = {}
    for name, M in raw.items():
        M = M - M.mean(0, keepdims=True) - M.mean(1, keepdims=True) + M.mean()     # offsets belong in h, not J
        out[name] = M / np.abs(M).max()
    return out


CHEM = chemistry_matrices()
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, (name, M) in zip(axes, CHEM.items()):
    vmax = np.abs(M).max()
    im = ax.imshow(M, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_xticks(range(20)); ax.set_xticklabels(list(AMINO_ACIDS), fontsize=6)
    ax.set_yticks(range(20)); ax.set_yticklabels(list(AMINO_ACIDS), fontsize=6)
    ax.set_title(f"{name}\n(red = pair favoured in a contact)")
    fig.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout(); plt.show()

In [ ]:
# ── 1.3 Sampling a family from the Potts model by Gibbs sampling ──
def sample_chemistry_family(rng, contacts, n_seqs=3000, n_sweeps=60, strength=1.8, noise=0.1):
    """Simulate an MSA from a Potts model whose couplings follow the three chemistry matrices.

    Same recipe as the toolkit's simulate_protein_family (fields h_i for conservation, Gibbs sampling with
    n_seqs independent chains), except that J_ij = strength · Σ_k w_ijk M_k + small noise instead of a random table.
    Returns msa (n_seqs × L ints), h (L × 20), J (L × L × 20 × 20), the mixing weights w, the wild type
    (fittest sampled sequence) and the mean Potts score after every sweep (to check that the sampler has settled).
    """
    L, q = contacts.shape[0], 20
    mats = np.stack(list(CHEM.values()))                                    # 3 × 20 × 20
    h = rng.normal(0, 1, (L, q)) * rng.uniform(0.3, 1.5, L)[:, None]        # stronger fields = more conserved
    J, w = np.zeros((L, L, q, q)), np.zeros((L, L, 3))
    for i, j in zip(*np.nonzero(np.triu(contacts, 1))):
        w[i, j] = w[j, i] = rng.dirichlet([0.5, 0.5, 0.5]) * rng.uniform(0.5, 1.5)   # which rules this contact uses
        J[i, j] = strength * np.tensordot(w[i, j], mats, axes=1) + rng.normal(0, noise, (q, q))
        J[j, i] = J[i, j].T
    neighbours = [np.flatnonzero(contacts[i]) for i in range(L)]
    x, rows, trace = rng.integers(0, q, (n_seqs, L)), np.arange(n_seqs), []
    for _ in range(n_sweeps):
        for i in rng.permutation(L):                                        # resample one column of all chains
            logits = np.tile(h[i], (n_seqs, 1))
            for j in neighbours[i]:
                logits += J[j, i][x[:, j]]                                  # adds J_ij(·, x_j)
            gumbel = -np.log(-np.log(rng.random((n_seqs, q))))              # Gumbel-max trick = softmax sampling
            x[rows, i] = np.argmax(logits + gumbel, axis=1)
        trace.append(potts_score(x, h, J).mean())
    wild_type = x[np.argmax(potts_score(x, h, J))].copy()
    return dict(msa=x, h=h, J=J, weights=w, wild_type=wild_type, trace=np.array(trace))


def seq_to_str(seq):
    """Integer-coded sequence → amino-acid string."""
    return "".join(AMINO_ACIDS[a] for a in seq)

In [ ]:
# ── 1.4 Sample 3,000 family members (a few seconds) ──
rng = np.random.default_rng(SEED + 1)
t0 = time.time()
family = sample_chemistry_family(rng, contacts, n_seqs=3000)
msa, h_true, J_true, wild_type = family["msa"], family["h"], family["J"], family["wild_type"]
print(f"SIMULATED alignment: {msa.shape[0]} sequences × {msa.shape[1]} positions, sampled in {time.time() - t0:.1f} s")
for k in range(4):
    print("  ", seq_to_str(msa[k]))
print("wild type:", seq_to_str(wild_type))
print("mean Potts score after sweeps 1, 10, 30, 60:", np.round(family["trace"][[0, 9, 29, 59]], 1))

🔍 **What just happened?** We drew 3,000 sequences, one Gibbs-sampling chain each
([Module 21](21_bayesian_learning_mcmc.ipynb)): repeatedly pick a position, compute the conditional probabilities of
the 20 residues given the current neighbours, and resample. The mean Potts score climbs during the first sweeps and then
levels off — the burn-in of Module 21 — so after 60 sweeps the chains are (approximately) samples from $P(x)$. The
sequences look unrelated at first glance, yet they are all "members of the family". The *wild type* is simply the
sequence with the highest score; we will mutate it in Section 6.

Now the two evolutionary signals. **Conservation** of column $i$ is measured by its entropy
$H_i=-\sum_a f_i(a)\log_2 f_i(a)$ (in bits; $\log_2 20=4.32$ for a column where all residues are equally common).
**Co-variation** of columns $i$ and $j$ shows up as joint frequencies $f_{ij}(a,b)$ that differ from the product of the
single-column frequencies $f_i(a)f_j(b)$.

In [ ]:
# ── 1.5 Column frequencies, conservation and one co-varying contact ──
def joint_frequencies(a, b):
    """20 × 20 table of how often residue pairs (a_n, b_n) occur together in two columns."""
    joint = np.zeros((20, 20)); np.add.at(joint, (a, b), 1)
    return joint / len(a)


def column_mi(a, b):
    """Mutual information (nats) between two columns: how far their joint table is from independence (Module 37)."""
    joint = joint_frequencies(a, b)
    outer, nz = np.outer(joint.sum(1), joint.sum(0)), joint > 0
    return (joint[nz] * np.log(joint[nz] / outer[nz])).sum()


freq = np.stack([np.bincount(msa[:, i], minlength=20) for i in range(L)]) / len(msa)        # L × 20
entropy = -(freq * np.log2(np.where(freq > 0, freq, 1))).sum(1)
i_c, j_c = 47, 23                     # one long-range contact, chosen because the effect is easy to see (✏️: try others)
assert long_range[i_c, j_c]
joint = joint_frequencies(msa[:, i_c], msa[:, j_c])
covar = joint - np.outer(freq[i_c], freq[j_c])
print(f"column entropy: {entropy.min():.2f} to {entropy.max():.2f} bits (uniform = {np.log2(20):.2f})")
print(f"example long-range contact: residue {i_c} with partner {j_c}, MI = {column_mi(msa[:, i_c], msa[:, j_c]):.3f} nats")

In [ ]:
# ── 1.6 Conservation, contacts and co-variation at a glance ──
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
cols = [OKABE_ITO[0] if s == "H" else OKABE_ITO[3] for s in ss]
axes[0].bar(range(L), entropy, color=cols)
axes[0].axhline(np.log2(20), color="k", ls="--", lw=0.8)
axes[0].set_xlabel("position"); axes[0].set_ylabel("entropy (bits)")
axes[0].set_title("conservation (blue = helix, orange = loop)\nSIMULATED family; dashed = uniform")
axes[1].imshow(contacts, cmap="Greys"); axes[1].plot(j_c, i_c, "o", mfc="none", mec=OKABE_ITO[1], ms=9)
axes[1].set_title(f"true contact map ({contacts.sum() // 2} pairs)\ncircle = pair ({i_c}, {j_c})")
axes[1].set_xlabel("position j"); axes[1].set_ylabel("position i")
vmax = np.abs(covar).max()
axes[2].imshow(covar, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[2].set_xticks(range(20)); axes[2].set_xticklabels(list(AMINO_ACIDS), fontsize=6)
axes[2].set_yticks(range(20)); axes[2].set_yticklabels(list(AMINO_ACIDS), fontsize=6)
axes[2].set_xlabel(f"residue at {j_c}"); axes[2].set_ylabel(f"residue at {i_c}")
axes[2].set_title(f"co-variation at contact ({i_c}, {j_c}): $f_{{ij}}-f_if_j$\n(red = seen together more than by chance)")
plt.tight_layout(); plt.show()

In [ ]:
# ── 1.7 The best residue at one position depends on its partner ──
print(f"contact ({i_c}, {j_c}): its true coupling mixes the rules with weights "
      f"{np.round(family['weights'][i_c, j_c], 2)} (hydrophobic, salt bridge, size)")
top2 = lambda p: ", ".join(f"{AMINO_ACIDS[a]} {p[a]:.2f}" for a in np.argsort(-p)[:2])
for b in np.argsort(-freq[j_c])[:4]:                     # the four most common residues at the partner position
    cond = joint[:, b] / joint[:, b].sum()               # p(x_i | x_j = b)
    print(f"  if position {j_c} is {AMINO_ACIDS[b]} ({(msa[:, j_c] == b).sum():4d} sequences): "
          f"most likely residues at {i_c}: {top2(cond)}")
print(f"  ignoring position {j_c}:                    most likely residues at {i_c}: {top2(freq[i_c])}")

🔍 **What just happened?** The left panel shows conservation varying from position to position (the simulator gave each
position its own field strength). The middle panel is the true contact map: the short diagonals are helix neighbours
($i\pm3$, $i\pm4$), the long off-diagonal stripes are helices packing against each other. The right panel zooms into
one long-range contact, residues 47 and 23, chosen because the effect is easy to see (its mutual information, a measure
defined in [Module 37](37_protein_sequences_language_models.ipynb), is printed in cell 1.5): some residue combinations
are clearly over-represented (red) and others avoided (blue). The printed lines make the key point. When position 23
carries a negatively charged residue (D or E), the most likely residues at position 47 are positively charged (R, K) —
a salt bridge; when position 23 carries a hydrophobic residue (A or V), the most likely residues at 47 are hydrophobic
(V, I); ignoring the partner, the best single guess at 47 (V) is right only about a quarter of the time. **The best
residue at one position depends on what sits at its partner position.** A model that predicts residues one column at a time (an independent-site model, or the
position-specific emissions of a profile HMM in [Module 20](20_hmm_learning.ipynb)) cannot capture this. A model that
looks at the *whole sequence* can.

**Tokens and embeddings.** A language model reads a sequence of **tokens** from a vocabulary. Ours has 21 tokens: the
20 amino acids (0–19) and a special `[MASK]` token (20) used in Section 4. The one-hot vectors of
[Module 22](22_neural_networks.ipynb) treat all residues as equally different. An **embedding** replaces each token by a
learned vector of $d$ numbers, so that residues that behave alike (I, L, V) can end up with similar vectors. In PyTorch
`nn.Embedding` is just a lookup table — mathematically, a one-hot vector multiplied by a weight matrix.

In [ ]:
# ── 1.8 A learned embedding = one-hot vector × weight matrix ──
MASK, VOCAB = 20, 21                       # tokens 0–19 = AMINO_ACIDS, token 20 = [MASK]
torch.manual_seed(SEED)
emb = nn.Embedding(VOCAB, 8)               # 21 tokens → 8-dimensional vectors (random now, learned in training)
tokens = torch.tensor(msa[0, :5])
one_hot = F.one_hot(tokens, VOCAB).float()
print("tokens", tokens.tolist(), "=", seq_to_str(msa[0, :5]))
print("shapes:", tuple(one_hot.shape), "@", tuple(emb.weight.shape), "→", tuple(emb(tokens).shape))
print("embedding lookup equals one-hot @ weights:", torch.allclose(emb(tokens), one_hot @ emb.weight))

✏️ **Your turn.** Repeat the co-variation analysis of cells 1.5–1.7 for a pair of positions that are **not** in contact
(pick one with `long_range[i, j] == False` and $|i-j|\ge 6$). How large is the largest $|f_{ij}-f_if_j|$ compared with
the contacting pair? Why is it not exactly zero? (Hint: 3,000 sequences are a finite sample.)

---
## 2 · Attention from scratch (NumPy)

> 📍 **Where we are.** We need a layer that lets every position gather information from *any* other position, with
> weights that depend on the content — "residue 12 should listen to residue 45 because they touch". That layer is
> **attention**. We write it in a dozen lines of NumPy, then study three of its properties.

**The idea.** Every position $i$ emits three vectors computed from its embedding $\mathbf x_i$ by learned matrices:
a **query** $\mathbf q_i=W_Q^\top\mathbf x_i$ ("what am I looking for?"), a **key** $\mathbf k_i=W_K^\top\mathbf x_i$
("what do I offer?") and a **value** $\mathbf v_i=W_V^\top\mathbf x_i$ ("what do I pass on?"). Position $i$ compares its
query with every key, turns the similarities into weights that sum to one, and outputs the weighted average of the
values. Stacking the vectors as rows of matrices $Q, K, V$:

$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\!\Big(\frac{QK^\top}{\sqrt{d_k}}\Big)V,
\qquad A_{ij}=\frac{\exp(\mathbf q_i\cdot\mathbf k_j/\sqrt{d_k})}{\sum_{j'}\exp(\mathbf q_i\cdot\mathbf k_{j'}/\sqrt{d_k})}.$$

*In words:* row $i$ of $QK^\top$ holds the dot products of query $i$ with all keys; dividing by $\sqrt{d_k}$ ($d_k$ =
length of a key) keeps them in a sensible range; the softmax turns each row into attention weights $A_{ij}\ge0$ that sum
to one over $j$; multiplying by $V$ replaces each position by a weighted average of all values. When queries, keys and
values all come from the same sequence it is called **self-attention**.

🧮 **Math refresher — softmax.** For numbers $s_1,\dots,s_n$, $\operatorname{softmax}(s)_j=e^{s_j}/\sum_k e^{s_k}$. Every
output is positive and they sum to 1, so they can be read as probabilities. Adding the same constant to all $s_j$ does
not change the result (it cancels between numerator and denominator), which is why code subtracts the maximum first
to avoid overflow. Large differences between the $s_j$ make the softmax "peaky" (almost all weight on the largest);
small differences make it nearly uniform. You met the same function as the multinomial logistic model and in the
log-sum-exp trick of [Module 19](19_hmm_inference.ipynb).

In [ ]:
# ── 2.1 Scaled dot-product attention in NumPy, and a 3-token example you can check by hand ──
def softmax_rows(S):
    """Softmax of every row (subtract the row maximum first, for numerical stability)."""
    E = np.exp(S - S.max(axis=-1, keepdims=True))
    return E / E.sum(axis=-1, keepdims=True)


def attention_numpy(Q, K, V, scale=None):
    """softmax(Q K^T · scale) V with scale = 1/sqrt(d_k) by default. Returns outputs and the weight matrix A."""
    scale = 1 / np.sqrt(Q.shape[-1]) if scale is None else scale
    A = softmax_rows(Q @ K.T * scale)
    return A @ V, A


q = np.array([[2.0, 0.0, 2.0, 0.0]])                    # one query, d_k = 4 so sqrt(d_k) = 2
K3 = np.array([[1.0, 0.0, 1.0, 0.0],                     # key of token 1
               [0.0, 1.0, 0.0, 1.0],                     # key of token 2
               [1.0, 1.0, 0.0, 0.0]])                    # key of token 3
V3 = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])      # 2-dimensional values
out, A = attention_numpy(q, K3, V3)
print("dot products q·k_j        :", q @ K3.T)
print("divided by sqrt(d_k) = 2  :", q @ K3.T / 2)
print("attention weights (sum 1) :", np.round(A, 3))
print("output = weighted values  :", np.round(out, 3))

🔍 **What just happened?** The query matches key 1 best (dot product 4), key 3 partly (2) and key 2 not at all (0).
After dividing by $\sqrt4=2$ the scores are 2, 0, 1; the softmax gives $e^2:e^0:e^1=7.39:1:2.72$, i.e. weights
0.665, 0.090 and 0.245. The output is $0.665\,\mathbf v_1+0.090\,\mathbf v_2+0.245\,\mathbf v_3=(0.910, 0.335)$.
Attention is nothing more than this — computed for every position at once by matrix products.

**Why divide by $\sqrt{d_k}$?** If the entries of $\mathbf q$ and $\mathbf k$ are independent with mean 0 and
variance 1, then $\mathbf q\cdot\mathbf k=\sum_{m=1}^{d_k}q_mk_m$ is a sum of $d_k$ terms of variance 1, so its variance is
$d_k$ (variances of independent terms add — [Module 02](02_probability_for_biologists.ipynb)). Long keys would give huge
scores, the softmax would put all its weight on one key, and its gradient would vanish (the saturation problem of
Module 22). Dividing by $\sqrt{d_k}$ brings the variance back to 1 whatever $d_k$ is. Let us check.

In [ ]:
# ── 2.2 Why divide by sqrt(d_k)? Peaky softmaxes for long keys ──
rng = np.random.default_rng(SEED + 2)
print(" d_k | var(q·k) | largest weight: unscaled  scaled | entropy (bits): unscaled  scaled   [10 keys; max 3.32]")
for d_k in [2, 8, 32, 128, 512]:
    qs, ks = rng.normal(size=(2000, d_k)), rng.normal(size=(2000, 10, d_k))   # 2,000 queries, 10 keys each
    s = np.einsum("nd,nkd->nk", qs, ks)
    stats = []
    for S in (s, s / np.sqrt(d_k)):
        A_ = softmax_rows(S)
        stats += [A_.max(1).mean(), -(A_ * np.log2(A_ + 1e-300)).sum(1).mean()]
    print(f"{d_k:4d} | {s.var():8.1f} |              {stats[0]:.2f}    {stats[2]:.2f} |"
          f"               {stats[1]:.2f}    {stats[3]:.2f}")

🔍 **What just happened?** The variance of the raw dot product grows in proportion to $d_k$ (about 2, 8, 32, … as
predicted). Without scaling, the largest of 10 weights approaches 1 and the entropy collapses towards 0 bits: every
query attends to a single key, whatever it learns. With scaling, the weights stay equally spread (the same entropy) at
every $d_k$. This is the reason for the $\sqrt{d_k}$ in Vaswani *et al.* (2017).

**Attention is a learned, soft k-nearest-neighbour average.** In [Module 08](08_knn_naive_bayes.ipynb) k-NN classified a
point by a vote of its $k$ nearest training points; for a continuous outcome the same idea, k-NN regression, averages
their outcomes with equal weights. Attention
also averages "values", but with smooth weights that decrease with distance. To see that this is more than an analogy,
take 1-D inputs and choose the query of point $x$ and the key of observation $x_j$ as

$$\mathbf q=\tfrac{1}{\tau}(x,\;1),\qquad \mathbf k_j=\tfrac{1}{\tau}\big(x_j,\;-\tfrac{x_j^2}{2}\big)
\;\Rightarrow\;\mathbf q\cdot\mathbf k_j=\frac{x\,x_j-x_j^2/2}{\tau^2}=-\frac{(x-x_j)^2}{2\tau^2}+\frac{x^2}{2\tau^2}.$$

*In words:* with these features the dot product is minus the squared distance between $x$ and $x_j$ (scaled by a
bandwidth $\tau$) plus a term $x^2/2\tau^2$ that is the same for every $j$ and therefore cancels in the softmax. The
attention weights become $A_j\propto\exp(-(x-x_j)^2/2\tau^2)$ — a Gaussian **kernel smoother** (the Nadaraya–Watson
estimator): a k-NN average with soft, distance-dependent weights instead of a hard cut-off. A transformer *learns* its
query and key features, so it learns which notion of "near" to use.

> 🔗 **Builds on** [Module 08 · k-NN](08_knn_naive_bayes.ipynb), Section 2: the hard vote of the $k$ nearest neighbours
> becomes a softmax-weighted vote over all neighbours, and "nearest" is measured in a learned feature space.

In [ ]:
# ── 2.3 Attention as a kernel smoother vs k-NN regression (SIMULATED expression along pseudotime) ──
rng = np.random.default_rng(SEED + 3)
x_obs = np.sort(rng.uniform(0, 10, 40))                         # 40 cells ordered along a developmental pseudotime
y_obs = np.sin(x_obs) + 0.1 * x_obs + rng.normal(0, 0.3, 40)    # expression of one gene (noisy)
x_new = np.linspace(0, 10, 200)


def knn_regress(x_query, x_train, y_train, k=5):
    """Module 08: average the outcomes of the k nearest training points (equal weights)."""
    idx = np.argsort(np.abs(x_query[:, None] - x_train[None, :]), axis=1)[:, :k]
    return y_train[idx].mean(axis=1)


tau = 0.5
Q_x = np.c_[x_new, np.ones_like(x_new)] / tau                   # queries (x, 1)/tau
K_x = np.c_[x_obs, -x_obs ** 2 / 2] / tau                       # keys (x_j, -x_j²/2)/tau
y_att, A_x = attention_numpy(Q_x, K_x, y_obs[:, None], scale=1.0)   # tau already sets the scale
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.4))
a1.scatter(x_obs, y_obs, s=14, color="0.4", label="observed cells")
a1.plot(x_new, knn_regress(x_new, x_obs, y_obs), color=OKABE_ITO[3], label="5-NN regression (Module 08)")
a1.plot(x_new, y_att[:, 0], color=OKABE_ITO[0], lw=2, label=f"attention smoother (τ = {tau})")
a1.set_xlabel("pseudotime"); a1.set_ylabel("expression"); a1.legend()
a1.set_title("SIMULATED: attention = soft nearest-neighbour average")
m = np.argmin(np.abs(x_new - 5.0))
near5 = np.argsort(np.abs(x_obs - x_new[m]))[:5]
a2.bar(x_obs, A_x[m], width=0.12, color=OKABE_ITO[0], label="attention weights")
a2.scatter(x_obs[near5], np.full(5, 0.2), marker="v", color=OKABE_ITO[3], label="5-NN: weight 0.2 each")
a2.set_xlabel("pseudotime of observed cell $x_j$"); a2.set_ylabel(f"weight for the query x = {x_new[m]:.1f}")
a2.set_xlim(2, 8); a2.set_title("one query: smooth weights vs a hard cut-off"); a2.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** Both curves follow the noisy expression trend; the attention smoother is smoother because
its weights fade gradually (right panel) while k-NN switches neighbours abruptly. Changing $\tau$ plays the role of
changing $k$: a small $\tau$ means few effective neighbours (wiggly, high variance), a large $\tau$ many (smooth, more
bias) — the bias–variance trade-off of [Module 23](23_model_selection.ipynb).

**A surprising property: attention ignores order.** Nothing in the formula refers to *where* a token sits. If we
shuffle the input positions, the outputs are shuffled in exactly the same way — self-attention is
**permutation-equivariant**. For a protein that is a disaster: "residue 12 touches residue 45" cannot be learned if the
model does not know which position is which.

In [ ]:
# ── 2.4 Self-attention is permutation-equivariant ──
rng = np.random.default_rng(SEED + 4)
X6 = rng.normal(size=(6, 8))                                     # 6 tokens with 8-dimensional embeddings
W_q, W_k, W_v = (rng.normal(size=(8, 8)) / np.sqrt(8) for _ in range(3))


def self_attention(X):
    """One self-attention layer with fixed random W_q, W_k, W_v."""
    return attention_numpy(X @ W_q, X @ W_k, X @ W_v)[0]


perm = rng.permutation(6)
print("permutation of the 6 positions:", perm)
print("self_attention(X[perm]) == self_attention(X)[perm] ?", np.allclose(self_attention(X6[perm]), self_attention(X6)[perm]))

**Positional encodings** break this symmetry by adding to each token embedding a vector that depends on its position
$p$. The original transformer used fixed sinusoids of different wavelengths,

$$\mathrm{PE}(p,2m)=\sin\!\big(p/10000^{2m/d}\big),\qquad \mathrm{PE}(p,2m+1)=\cos\!\big(p/10000^{2m/d}\big),$$

*In words:* dimension pair $(2m, 2m+1)$ of the encoding is a point moving around a circle as $p$ increases, fast for
small $m$ and slowly for large $m$ — like the hands of a clock with many hands — so every position gets a unique code
and nearby positions get similar codes. The alternative is to **learn** one vector per position (our model in Section 3
does this, since all our sequences have the same length 68); ESM-2 uses *rotary* encodings, which rotate queries and keys
by position-dependent angles so that $\mathbf q_i\cdot\mathbf k_j$ depends on the offset $i-j$.

In [ ]:
# ── 2.5 Sinusoidal positional encodings break the symmetry ──
def sinusoidal_encoding(n_pos, d):
    """Vaswani et al. (2017): PE[p, 2m] = sin(p / 10000^(2m/d)), PE[p, 2m+1] = cos(p / 10000^(2m/d))."""
    p, m2 = np.arange(n_pos)[:, None], np.arange(0, d, 2)[None, :]
    PE = np.zeros((n_pos, d))
    PE[:, 0::2], PE[:, 1::2] = np.sin(p / 10000 ** (m2 / d)), np.cos(p / 10000 ** (m2 / d))
    return PE


PE6 = sinusoidal_encoding(6, 8)
print("with positions added, still equivariant?",
      np.allclose(self_attention(X6[perm] + PE6), self_attention(X6 + PE6)[perm]))
PE = sinusoidal_encoding(L, 64)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.2))
im = a1.imshow(PE.T, aspect="auto", cmap="RdBu_r"); fig.colorbar(im, ax=a1, shrink=0.8)
a1.set_xlabel("position p"); a1.set_ylabel("encoding dimension"); a1.set_title("sinusoidal encodings (L = 68, d = 64)")
a2.plot(np.arange(L), PE @ PE[30], color=OKABE_ITO[0])
a2.axvline(30, color="k", ls=":", lw=0.8)
a2.set_xlabel("position p"); a2.set_ylabel("PE(p) · PE(30)")
a2.set_title("similar codes for nearby positions")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Adding position codes makes the outputs depend on order: the equivariance check now fails
(`False`), as it must. The heat map shows the clock-like code: low dimensions oscillate fast, high dimensions slowly.
The dot product between the code of position 30 and all others peaks at 30 and decays with distance, so attention can
easily express "look at neighbours" — and, with learned $W_Q, W_K$, "look 4 positions ahead" or "look at position 45".

⚠️ **Pitfalls.** (1) Attention weights say where a layer *reads* information from; they are not by themselves an
explanation of a prediction, and they are certainly not evidence of a physical or regulatory interaction (Section 5
checks them against ground truth instead of trusting them). (2) Sequences of different lengths are padded to a common
length; the padding positions must be **masked** out of the softmax (scores set to $-\infty$), otherwise real tokens
attend to padding. All our sequences have length 68, so we need no padding.

✏️ **Your turn.** In cell 2.3 try `tau = 0.15` and `tau = 2.0`. Which value over-fits the noise and which
over-smooths the bump near pseudotime 2? Which $k$ in `knn_regress` gives a similar curve to each?

---
## 3 · The transformer block (PyTorch)

> 📍 **Where we are.** One attention layer gathers information; a **transformer** stacks attention with a few other
> ingredients that make deep networks trainable. We build it in PyTorch with the `nn.Module` pattern of
> [Module 22](22_neural_networks.ipynb), check it against PyTorch's own implementation, and count its parameters.

**The four ingredients.**

1. **Multi-head attention.** Instead of one attention with $d$-dimensional queries, use $H$ heads, each with its own
   $W_Q^{(h)},W_K^{(h)},W_V^{(h)}$ projecting to $d_k=d/H$ dimensions. Different heads can attend to different partners
   (one to helix neighbours, another to residues across the bundle). Their outputs are concatenated and mixed by $W_O$.
2. **Residual connections.** Each sub-layer *adds* its output to its input, $\mathbf x\leftarrow\mathbf x+f(\mathbf x)$,
   so information and gradients can flow straight through a deep stack (the vanishing-gradient problem of Module 22).
3. **Layer normalisation.** Before each sub-layer the vector of every position is standardised (🧮 below).
4. **A position-wise MLP.** A two-layer network applied to every position separately, which transforms what attention
   gathered.

With "pre-LayerNorm" ordering (used by most modern models, including ESM-2), one block computes

$$\mathbf x\;\leftarrow\;\mathbf x+\operatorname{MHA}\big(\operatorname{LN}(\mathbf x)\big),\qquad
\mathbf x\;\leftarrow\;\mathbf x+\operatorname{MLP}\big(\operatorname{LN}(\mathbf x)\big).$$

*In words:* normalise, let every position gather information from the others, add the result to what was there;
normalise again, transform every position with a small MLP, add again.

🧮 **Math refresher — LayerNorm.** For one position's vector $\mathbf x\in\mathbb R^d$,
$\operatorname{LN}(\mathbf x)=\boldsymbol\gamma\odot(\mathbf x-\bar x)/\sqrt{s^2+\epsilon}+\boldsymbol\beta$, where $\bar x$
and $s^2$ are the mean and variance of its $d$ entries, $\odot$ multiplies entry by entry, and the scale
$\boldsymbol\gamma$ and shift $\boldsymbol\beta$ are learned. It is the z-score of
[Module 01](01_python_numpy_for_biologists.ipynb) applied across features (not across samples), which keeps the numbers
flowing through the network in a stable range.

In [ ]:
# ── 3.1 Multi-head self-attention as a PyTorch module ──
class MultiHeadSelfAttention(nn.Module):
    """H attention heads in parallel; each works in its own d_model/H-dimensional subspace."""

    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads, self.d_k = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)        # W_Q, W_K, W_V of all heads in one matrix
        self.out = nn.Linear(d_model, d_model)            # W_O mixes the concatenated heads

    def forward(self, x):
        B, T, D = x.shape                                   # batch, positions, features
        q, k, v = self.qkv(x).view(B, T, 3, self.n_heads, self.d_k).permute(2, 0, 3, 1, 4)   # each B×H×T×d_k
        A = torch.softmax(q @ k.transpose(-2, -1) / math.sqrt(self.d_k), dim=-1)            # B×H×T×T
        z = (A @ v).transpose(1, 2).reshape(B, T, D)       # weighted values, heads concatenated
        return self.out(z), A

In [ ]:
# ── 3.2 Check: our module reproduces PyTorch's built-in nn.MultiheadAttention ──
torch.manual_seed(SEED)
ours = MultiHeadSelfAttention(16, 4)
ref = nn.MultiheadAttention(16, 4, batch_first=True)
with torch.no_grad():                                      # give both modules the same weights
    ref.in_proj_weight.copy_(ours.qkv.weight); ref.in_proj_bias.copy_(ours.qkv.bias)
    ref.out_proj.weight.copy_(ours.out.weight); ref.out_proj.bias.copy_(ours.out.bias)
x = torch.randn(2, 7, 16)                                  # 2 sequences, 7 positions, 16 features
y_ours, A_ours = ours(x)
y_ref, A_ref = ref(x, x, x, need_weights=True, average_attn_weights=False)
print("max |difference| of outputs          :", f"{(y_ours - y_ref).abs().max().item():.1e}")
print("max |difference| of attention weights:", f"{(A_ours - A_ref).abs().max().item():.1e}")

🔍 **What just happened?** The differences are zero (on some machines they may be at the level of floating-point
rounding, around $10^{-7}$): our dozen lines compute exactly what PyTorch's optimised module computes. Now we wrap attention into a block and the block into a
small encoder. (We give it a name, `TinyEncoder`, that cannot be confused with any library model.)

In [ ]:
# ── 3.3 A transformer block and a tiny encoder ──
class TransformerBlock(nn.Module):
    """Pre-LayerNorm block: x + MHA(LN(x)), then x + MLP(LN(x))."""

    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads)
        self.mlp = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x):
        a, A = self.attn(self.ln1(x))
        x = x + a                                          # residual connection around attention
        x = x + self.mlp(self.ln2(x))                      # residual connection around the MLP
        return x, A


class TinyEncoder(nn.Module):
    """Token + learned position embeddings → transformer blocks → logits over the 20 residues at every position."""

    def __init__(self, n_pos, d_model=64, n_heads=4, n_layers=2, d_ff=128, vocab=VOCAB, n_out=20):
        super().__init__()
        self.tok, self.pos = nn.Embedding(vocab, d_model), nn.Embedding(n_pos, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.ln, self.head = nn.LayerNorm(d_model), nn.Linear(d_model, n_out)

    def forward(self, tokens):
        """tokens: B × T ints → logits (B×T×20), attention maps (B×layers×heads×T×T), embeddings (B×T×d)."""
        x = self.tok(tokens) + self.pos.weight[: tokens.shape[1]]
        maps = []
        for block in self.blocks:
            x, A = block(x)
            maps.append(A)
        z = self.ln(x)
        return self.head(z), torch.stack(maps, dim=1), z

In [ ]:
# ── 3.4 Count the parameters by hand and by PyTorch ──
torch.manual_seed(SEED)
model = TinyEncoder(n_pos=L)                    # d_model = 64, 4 heads, 2 layers, MLP width 128
d, d_ff = 64, 128
by_hand = {"token embedding (21 × 64)": VOCAB * d,
           "position embedding (68 × 64)": L * d,
           "2 × attention (W_QKV, W_O + biases)": 2 * (3 * d * d + 3 * d + d * d + d),
           "2 × MLP (64 → 128 → 64 + biases)": 2 * (d * d_ff + d_ff + d_ff * d + d),
           "2 × two LayerNorms (γ and β)": 2 * 2 * 2 * d,
           "final LayerNorm + output head (64 → 20)": 2 * d + d * 20 + 20}
for name, n in by_hand.items():
    print(f"  {name:42s} {n:7,d}")
print(f"  {'total by hand':42s} {sum(by_hand.values()):7,d}")
print(f"  {'total according to PyTorch':42s} {sum(p.numel() for p in model.parameters()):7,d}")

In [ ]:
# ── 3.5 The architecture at a glance ──
from matplotlib.patches import FancyBboxPatch
fig, ax = plt.subplots(figsize=(12, 2.6))
boxes = [("68 tokens\n(some = [MASK])", "#eeeeee"), ("token + position\nembedding\n(68 × 64)", "#dbe9f6"),
         ("block 1\nLN → attention → +\nLN → MLP → +", "#cfe8d5"), ("block 2\nLN → attention → +\nLN → MLP → +", "#cfe8d5"),
         ("LN → linear head\n20 logits\nper position", "#fde2c8"), ("cross-entropy\non the masked\npositions", "#f6d5e5")]
width, gap = 1.7, 0.5
for k, (text, colour) in enumerate(boxes):
    x0 = k * (width + gap)
    ax.add_patch(FancyBboxPatch((x0, 0.22), width, 0.56, boxstyle="round,pad=0.04", fc=colour, ec="0.3"))
    ax.text(x0 + width / 2, 0.5, text, ha="center", va="center", fontsize=8.5)
    if k:
        ax.annotate("", xy=(x0 - 0.06, 0.5), xytext=(x0 - gap + 0.06, 0.5), arrowprops=dict(arrowstyle="->", color="0.3"))
ax.text(3 * (width + gap) - gap / 2, 0.04, "each block: 4 attention heads; the attention maps (2 layers × 4 heads × 68 × 68) "
        "are kept for Section 5", ha="center", fontsize=8, color="0.3")
ax.set_xlim(-0.2, len(boxes) * (width + gap) - gap + 0.2); ax.set_ylim(0, 1); ax.axis("off")
ax.set_title("TinyEncoder: a 2-layer transformer encoder for masked-residue prediction")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The hand count and PyTorch agree. Note where the parameters live: each attention layer has
$4d^2+4d=16{,}640$ parameters whatever the sequence length, because the same $W_Q,W_K,W_V,W_O$ are applied at every
position (weight sharing, like the convolution filters of Module 22). Only the learned position embedding grows with
$L$. ESM-2 uses the same building blocks with $d$ up to 5,120, up to 48 layers and up to 15 billion parameters — our
model is about five orders of magnitude smaller.

✏️ **Your turn.** Without running anything: how many parameters would `TinyEncoder(n_pos=68, d_model=128, n_heads=8)`
have? Which terms grow like $d^2$ and which like $d$? Then check with `sum(p.numel() for p in ...)`.

### 📄 Research connection — Attention and transformers

- **What the field does.** Attention was introduced for machine translation by Bahdanau, Cho & Bengio (2015) as a way for a recurrent
    decoder to look back at all input words. Vaswani *et al.* (2017) dropped recurrence altogether ("Attention is all you
    need"): stacks of multi-head self-attention, residual connections, LayerNorm and MLPs — the **transformer**. BERT
    (Devlin *et al.* 2019) pre-trained a transformer *encoder* by masked-token prediction and showed that the learned
    representations transfer to many tasks; GPT-style models pre-train a *decoder* by predicting the next token.
- **What this notebook builds.** A from-scratch NumPy attention (with the $\sqrt{d_k}$ scaling derived), a PyTorch multi-head attention
    verified against `nn.MultiheadAttention`, and a 2-layer pre-LayerNorm encoder with about 75,000 parameters — the
    same block structure as BERT and ESM-2, at toy scale.
- **Where the frontier is.** Attention costs memory and time proportional to $L^2$, which limits context length (Section 8). Active
    directions (2023–2026) include efficient attention kernels, sparse and linear attention, and non-attention sequence
    models (state-space and long-convolution models such as Hyena), as well as understanding *what* individual heads
    compute (mechanistic interpretability). How much of a transformer's success comes from architecture versus scale and
    data is still debated.
- **Read next.** Vaswani *et al.* 2017, *Attention is all you need* (NeurIPS); Devlin *et al.* 2019, *BERT* (NAACL);
    Bahdanau, Cho & Bengio 2015 (ICLR).

---
## 4 · Self-supervised pre-training: masked language modelling

> 📍 **Where we are.** We have a model that maps a sequence to 20 scores per position. To train it we need a task.
> There are no labels — only sequences — so the sequences must supervise themselves: hide some residues and ask the model
> to predict them. This is **masked language modelling** (MLM), BERT's recipe and ESM's.

**The objective.** For a sequence $x$, choose a random set $M$ of about 15 % of the positions, corrupt them to get
$\tilde x$, and minimise

$$\mathcal L(\theta)=-\frac{1}{|M|}\sum_{i\in M}\log p_\theta\big(x_i\mid\tilde x\big).$$

*In words:* the loss is the cross-entropy ([Module 09](09_logistic_regression.ipynb), [Module 22](22_neural_networks.ipynb))
between the true residue and the model's predicted distribution, averaged over the hidden positions only. Minimising it is
maximum likelihood ([Module 05](05_likelihood_mle_map.ipynb)) for the conditional distributions of residues given their
context. BERT corrupts the chosen positions in three ways: 80 % become `[MASK]`, 10 % a random residue and 10 % stay
unchanged, so that the model cannot rely on `[MASK]` being present at prediction time.

> 🔗 **Builds on** [Module 22 · Neural networks](22_neural_networks.ipynb), Sections 4 and 7: the same cross-entropy loss,
> backpropagation and PyTorch training loop — only the input (a corrupted sequence) and the targets (the hidden residues)
> are new. And [Module 05](05_likelihood_mle_map.ipynb): minimising cross-entropy is maximising a likelihood.

**Why this should learn co-variation.** Section 1 showed that the best guess for a residue depends on its contact
partners. A model that wants to lower the loss on masked position $i$ must therefore find, among the 67 other positions,
the ones that are informative about $i$ — its 3-D neighbours — and read their residues. Attention is exactly the tool
for this. Nothing about structure is given to the model; it can only discover it from the statistics of the family.

In [ ]:
# ── 4.1 Train/test split and BERT-style masking ──
rng = np.random.default_rng(SEED + 5)
order = rng.permutation(len(msa))
train_idx, test_idx = order[:2500], order[2500:]
X_train, X_test = torch.tensor(msa[train_idx]), torch.tensor(msa[test_idx])


def mask_tokens(x, rate=0.15, gen=None):
    """BERT corruption: choose `rate` of the positions; 80 % → [MASK], 10 % → random residue, 10 % unchanged.
    Returns the corrupted tokens and the boolean matrix of chosen positions (where the loss is computed)."""
    chosen = torch.rand(x.shape, generator=gen) < rate
    r = torch.rand(x.shape, generator=gen)
    x_in = x.clone()
    x_in[chosen & (r < 0.8)] = MASK
    swap = chosen & (r >= 0.8) & (r < 0.9)
    x_in[swap] = torch.randint(0, 20, (int(swap.sum()),), generator=gen)
    return x_in, chosen


def show_tokens(t):
    """Token tensor → string, with '·' for [MASK]."""
    return "".join("·" if a == MASK else AMINO_ACIDS[a] for a in t.tolist())


demo_in, demo_chosen = mask_tokens(X_train[:1], gen=torch.Generator().manual_seed(SEED))
print("original :", show_tokens(X_train[0]))
print("corrupted:", show_tokens(demo_in[0]))
print("loss at  :", "".join("^" if c else " " for c in demo_chosen[0].tolist()))
print(f"{len(train_idx)} training and {len(test_idx)} held-out sequences")

In [ ]:
# ── 4.2 Masked-language-model pre-training (≈ 30–60 s on a 2-core CPU) ──
torch.manual_seed(SEED)
model = TinyEncoder(n_pos=L)
STEPS, BATCH, LR = 900, 64, 4e-3
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=STEPS, pct_start=0.1)   # warm up, then decay
val_x = X_test[:300]
val_in, val_chosen = mask_tokens(val_x, gen=torch.Generator().manual_seed(SEED + 1))   # fixed validation corruption
val_hidden = val_chosen & (val_in == MASK)                 # evaluate where the residue is really hidden
trn_x = X_train[:300]                                      # the same evaluation on 300 TRAINING sequences
trn_in, trn_chosen = mask_tokens(trn_x, gen=torch.Generator().manual_seed(SEED + 2))
trn_hidden = trn_chosen & (trn_in == MASK)
history, t0 = [], time.time()
for step in range(STEPS):
    batch = X_train[torch.randint(0, len(X_train), (BATCH,))]
    x_in, chosen = mask_tokens(batch)                      # a fresh random corruption every step
    logits, _, _ = model(x_in)
    loss = F.cross_entropy(logits[chosen], batch[chosen])  # only the chosen positions count
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    if step % 25 == 0 or step == STEPS - 1:
        model.eval()
        with torch.no_grad():
            v_loss = F.cross_entropy(model(val_in)[0][val_hidden], val_x[val_hidden]).item()
            t_loss = F.cross_entropy(model(trn_in)[0][trn_hidden], trn_x[trn_hidden]).item()
        model.train()
        history.append((step, t_loss, v_loss))           # both evaluated the same way: [MASK]ed residues only
history = np.array(history)
print(f"{STEPS} steps in {time.time() - t0:.0f} s; loss on [MASK]ed residues: "
      f"training sequences {history[-1, 1]:.3f}, held-out sequences {history[-1, 2]:.3f} nats")

In [ ]:
# ── 4.3 Learning curve against two baselines ──
counts_train = np.stack([np.bincount(msa[train_idx][:, i], minlength=20) for i in range(L)])
log_p_indep = np.log((counts_train + 0.5) / (counts_train + 0.5).sum(1, keepdims=True))   # L × 20, pseudocount 0.5
pos_hidden = np.nonzero(val_hidden.numpy())
ce_indep = -log_p_indep[pos_hidden[1], val_x.numpy()[pos_hidden]].mean()   # same hidden residues, no context

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(history[:, 0], history[:, 1], color="0.6", lw=1.5, label="training sequences")
ax.plot(history[:, 0], history[:, 2], color=OKABE_ITO[0], lw=2, label="held-out sequences")
ax.axhline(np.log(20), color="k", ls="--", lw=0.8, label="uniform guess: log 20")
ax.axhline(ce_indep, color=OKABE_ITO[3], ls="--", lw=1.2, label="independent-site model (no context)")
ax.set_xlabel("training step"); ax.set_ylabel("cross-entropy on masked residues (nats)")
ax.set_title("SIMULATED family: masked-residue loss during pre-training"); ax.legend()
plt.tight_layout(); plt.show()
print(f"held-out loss: transformer {history[-1, 2]:.3f} | independent-site {ce_indep:.3f} | uniform {np.log(20):.3f}")

🔍 **What just happened?** Within about 150–200 steps the held-out loss falls from the uniform guess to the
independent-site line: the transformer has learned the column frequencies, which is what a profile (a PSSM, the emissions
of a profile HMM) would give. Then it keeps going *below* that line: it has started to use the context. Both curves are
evaluated the same way — on the `[MASK]`ed residues of one fixed corruption — and they lie almost on top of each other,
ending only about 0.03 nats apart (printed in cell 4.2): the model has barely over-fitted. (Comparing the raw
training-batch loss with the held-out loss would mislead, because the training loss also includes the easy unchanged
positions and the randomised ones.) A fresh random masking at every step acts like data augmentation; weight decay and a
short schedule help too.

How good is this? Because we simulated the data, we can compute the best possible predictor: the **true conditional
distribution** of a residue given all the others under the Potts model,

$$p(x_i=a\mid x_{-i})=\frac{\exp\big(h_i(a)+\sum_{j}J_{ij}(a,x_j)\big)}{\sum_{b}\exp\big(h_i(b)+\sum_{j}J_{ij}(b,x_j)\big)}.$$

*In words:* with the true fields and couplings, the probability of residue $a$ at a hidden position is a softmax of its
field plus its couplings to the residues actually present at the other positions. (This "conditional is a softmax of
local terms" is the categorical version of Module 18's "conditional of a Gaussian is a linear regression".) No model
can beat it on average. We compare four predictors on held-out sequences, hiding **one** position at a time.

🧮 **Math refresher — perplexity.** If a model gives the true residues an average log-probability of $-\bar\ell$ (the
cross-entropy in nats), its **perplexity** is $e^{\bar\ell}$: the number of equally likely residues it is effectively
choosing among. A uniform guess over 20 amino acids has perplexity exactly 20; a perfect predictor has perplexity 1.

In [ ]:
# ── 4.4 🧪 Ground-truth check: masked-residue accuracy and perplexity vs the oracle ──
@torch.no_grad()
def single_site_log_probs(model, X):
    """log p(x_i = a | all other residues) for every sequence, position i and residue a (one [MASK] at a time)."""
    model.eval()
    out = torch.zeros(X.shape[0], X.shape[1], 20)
    for i in range(X.shape[1]):
        x_in = X.clone(); x_in[:, i] = MASK
        logits, _, _ = model(x_in)
        out[:, i] = torch.log_softmax(logits[:, i], dim=-1)
    return out.numpy()


def potts_conditional_log_probs(X, h, J):
    """The same quantity under the TRUE Potts model: softmax over a of h_i(a) + Σ_j J_ij(a, x_j)."""
    logits = np.repeat(h[None], len(X), axis=0)                       # n × L × 20
    for j in range(X.shape[1]):
        logits += J[:, j][:, :, X[:, j]].transpose(2, 0, 1)            # adds J_ij(a, x_j) for every i at once
    return logits - lse(logits, axis=-1, keepdims=True)


X_eval = X_test[:120]                                                  # 120 held-out sequences × 68 positions
xe = X_eval.numpy()
predictors = {"uniform (20 letters)": np.full((len(xe), L, 20), -np.log(20)),
              "independent-site (column frequencies)": np.repeat(log_p_indep[None], len(xe), axis=0),
              "TinyEncoder (masked LM)": single_site_log_probs(model, X_eval),
              "true Potts conditional (oracle)": potts_conditional_log_probs(xe, h_true, J_true)}
r_, c_ = np.arange(len(xe))[:, None], np.arange(L)[None, :]
results = pd.DataFrame([(name, np.exp(-lp[r_, c_, xe].mean()), (lp.argmax(-1) == xe).mean())
                        for name, lp in predictors.items()], columns=["predictor", "perplexity", "accuracy"])
print(results.round(3).to_string(index=False))
lppl = np.log(results["perplexity"].to_numpy())
print(f"\nshare of the context gain (independent → oracle) captured by the transformer: {(lppl[1] - lppl[2]) / (lppl[1] - lppl[3]):.0%}")

🔍 **What just happened?** The ladder is exactly what the biology predicts. A uniform guess has perplexity 20; knowing
the column frequencies (conservation) brings it to about 12.9; the true model, which also knows the co-variation,
reaches about 9.9 — that gap is the information carried by context. Our transformer lands in between, at about 11.1,
and its accuracy rises accordingly (from 0.22 to 0.26, against 0.29 for the oracle): in well under a minute of training on
2,500 sequences it has captured almost 60 % of the information that context can provide, without ever being told which
residues touch. (The numbers are reproducible because everything is seeded.)

⚠️ **Pitfall — evaluation leakage.** We evaluated on held-out sequences of the same family, drawn independently.
In real data, held-out proteins are usually *related* to training proteins (homologues share most of their sequence),
so a model can look good by recognising relatives. Honest benchmarks split by sequence identity or by family — see
[Module 42](42_trustworthy_ml.ipynb).

✏️ **Your turn.** Retrain with `rate=0.4` in `mask_tokens` (inside the training loop only). The task becomes harder
(less context per prediction): does the single-site perplexity get worse or better after 900 steps? Why might a higher
masking rate sometimes help a small model?

---
## 5 · 🧪 What did it learn? Structure

> 📍 **Where we are.** The model predicts masked residues better than the independent-site model, so it must be reading
> informative partners. Are those partners the true 3-D contacts? Rao *et al.* (2021) found that, in large protein
> language models, some attention heads attend from a residue to its contact partners, so that a simple combination of
> attention maps predicts contacts. We test the same idea where the truth is known.

**Recipe (as in Rao *et al.*).** For each layer and head, average the attention maps over many sequences,
**symmetrise** ($A+A^\top$: contact is symmetric, attention is not) and apply the **average product correction**,

$$\mathrm{APC}(S)_{ij}=S_{ij}-\frac{\bar S_{i\cdot}\,\bar S_{\cdot j}}{\bar S_{\cdot\cdot}},$$

*In words:* subtract from every score what would be expected from the overall "popularity" of positions $i$ and $j$
(row and column means $\bar S_{i\cdot},\bar S_{\cdot j}$ divided by the overall mean $\bar S_{\cdot\cdot}$). Some positions
receive attention from everybody — a background that says nothing about pairs. APC was introduced for mutual information
by Dunn, Wahl & Gloor (2008) and is used throughout contact prediction
([Module 37](37_protein_sequences_language_models.ipynb)).

**Scoring.** Rank all pairs with $|i-j|\ge6$ by their score and report the **precision of the top $L/2=34$ pairs**:
the fraction that are true contacts. A random ranking would score the base rate printed in cell 1.1 (about 2 %).

In [ ]:
# ── 5.1 Helpers: APC and precision of the top-ranked pairs ──
def apc_correct(S):
    """Average product correction (Dunn et al. 2008): S_ij − mean_i · mean_j / overall mean (diagonal ignored)."""
    S = S.copy(); np.fill_diagonal(S, 0)
    n = len(S)
    row = S.sum(1) / (n - 1)
    return S - np.outer(row, row) / (S.sum() / (n * (n - 1)))


def precision_at_k(scores, true_contacts, k, min_sep=6):
    """Fraction of true contacts among the k highest-scoring pairs with |i − j| ≥ min_sep."""
    iu = np.triu_indices(len(scores), min_sep)
    top = np.argsort(-scores[iu])[:k]
    return true_contacts[iu][top].mean()


K_TOP = L // 2
print(f"we score the top L/2 = {K_TOP} pairs with |i − j| ≥ 6; "
      f"{long_range.sum() // 2} of the {n_pairs} candidate pairs are true contacts (base rate {long_range.sum() / 2 / n_pairs:.3f})")

In [ ]:
# ── 5.2 Attention maps of held-out sequences vs the true contact map ──
model.eval()
with torch.no_grad():
    _, attn, _ = model(X_test[:300])                       # 300 × layers × heads × L × L
attn = attn.mean(0).numpy()                                 # average over sequences → layers × heads × L × L
head_prec = np.array([[precision_at_k(apc_correct(a + a.T), contacts, K_TOP) for a in layer] for layer in attn])
avg = attn.mean(axis=(0, 1))
attn_scores = apc_correct(avg + avg.T)                      # all heads and layers averaged, symmetrised, APC
print("precision of the top L/2 pairs, head by head (rows = layers, columns = heads):")
print(pd.DataFrame(head_prec, index=["layer 1", "layer 2"], columns=[f"head {k + 1}" for k in range(4)]).round(2))
print(f"\nall heads averaged + APC : {precision_at_k(attn_scores, contacts, K_TOP):.2f}")
print(f"without APC              : {precision_at_k(avg + avg.T, contacts, K_TOP):.2f}")

**A second probe: the categorical Jacobian.** Attention says where a layer reads from, not how much the prediction
depends on what it reads. A more direct question: *if I mutate residue $j$, how much does the model's predicted
distribution at residue $i$ change?* Feed the wild-type sequence (no masking), then every one of its $L\times20$ single
mutants, and record the change in the output logits:

$$\mathcal J_{ij}(a,b)=\operatorname{logit}_{i,a}\big(x^{(j\to b)}\big)-\operatorname{logit}_{i,a}(x).$$

*In words:* entry $(a,b)$ of block $(i,j)$ is how much the model's score for residue $a$ at position $i$ moves when
position $j$ is changed to residue $b$. If the model has learned the Potts model, $\mathcal J_{ij}(a,b)$ should resemble the
coupling $J_{ij}(a,b)$ (up to row/column offsets), so the size of each block (its Frobenius norm after removing
row and column means) should be large for contacts. This probe was proposed for protein language models by Zhang
*et al.* (2024); it needs no attention maps at all and works for any architecture.

In [ ]:
# ── 5.3 The categorical Jacobian of the wild type ──
@torch.no_grad()
def categorical_jacobian(model, seq):
    """Jac[i, a, j, b] = change of logit (i, a) when residue j of `seq` is replaced by b (no masking)."""
    model.eval()
    base = torch.tensor(seq)[None]
    logits0, _, _ = model(base)
    variants = base.repeat(L * 20, 1)
    for j in range(L):
        variants[j * 20:(j + 1) * 20, j] = torch.arange(20)        # all 20 residues at position j
    logits, _, _ = model(variants)
    return (logits - logits0).numpy().reshape(L, 20, L, 20).transpose(2, 3, 0, 1)   # → (i, a, j, b)


def block_norms(Jx):
    """Frobenius norm of every 20 × 20 block (axes i, a, j, b) after removing row and column means; symmetrised."""
    Jc = Jx - Jx.mean(axis=1, keepdims=True) - Jx.mean(axis=3, keepdims=True) + Jx.mean(axis=(1, 3), keepdims=True)
    N_ = np.sqrt((Jc ** 2).sum(axis=(1, 3)))
    N_ = (N_ + N_.T) / 2
    np.fill_diagonal(N_, 0)
    return N_


jac_scores = apc_correct(block_norms(categorical_jacobian(model, wild_type)))
true_scores = block_norms(J_true.transpose(0, 2, 1, 3))            # the same statistic for the TRUE couplings
print(f"categorical Jacobian + APC : precision@L/2 = {precision_at_k(jac_scores, contacts, K_TOP):.2f}")
print(f"true couplings (sanity)    : precision@L/2 = {precision_at_k(true_scores, contacts, K_TOP):.2f}")

In [ ]:
# ── 5.4 Predicted vs true contacts ──
def plot_contact_prediction(ax, scores, title, k=K_TOP, min_sep=6):
    """Grey = true contacts (both triangles); upper triangle: top-k predicted pairs, blue = correct, red = wrong."""
    ax.imshow(contacts, cmap="Greys", alpha=0.35)
    iu = np.triu_indices(L, min_sep)
    top = np.argsort(-scores[iu])[:k]
    ii, jj = iu[0][top], iu[1][top]
    ok = contacts[ii, jj]
    ax.scatter(jj[ok], ii[ok], s=14, color=OKABE_ITO[0], label="correct")
    ax.scatter(jj[~ok], ii[~ok], s=14, marker="x", color=OKABE_ITO[1], label="wrong")
    ax.set_title(f"{title}\nprecision@L/2 = {ok.mean():.2f}"); ax.set_xlabel("position j"); ax.set_ylabel("position i")


fig, axes = plt.subplots(1, 3, figsize=(12, 3.9))
plot_contact_prediction(axes[0], attn_scores, "attention (all heads, APC)")
plot_contact_prediction(axes[1], jac_scores, "categorical Jacobian (APC)")
axes[1].legend(loc="lower left", fontsize=7)
im = axes[2].imshow(head_prec, cmap="viridis", vmin=0, vmax=1)
for (r, c), v in np.ndenumerate(head_prec):
    axes[2].text(c, r, f"{v:.2f}", ha="center", va="center", color="w" if v < 0.5 else "k")
axes[2].set_xticks(range(4)); axes[2].set_xticklabels([f"head {k + 1}" for k in range(4)])
axes[2].set_yticks(range(2)); axes[2].set_yticklabels(["layer 1", "layer 2"])
axes[2].set_title("precision@L/2 of single heads"); fig.colorbar(im, ax=axes[2], shrink=0.6)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The averaged attention map ranks true contacts far above chance — 0.65 precision against a
base rate of 0.019 — and the categorical Jacobian does almost as well (0.62). The model was never shown a structure:
the contact map emerged from learning to fill in blanks. Four further observations:

* **No single head is enough.** The best single head reaches only 0.47, yet the average of all eight reaches 0.65: each
  head tracks a different subset of the contacts, and averaging also cancels head-specific noise. (Rao *et al.* found
  that in large models some heads are much better contact predictors than others, and combined them with a sparse
  logistic regression.)
* **APC changed nothing here** (0.65 with and without): the attention of this small model has little position-specific
  background. In large models, and for the mutual information of Module 37, it matters a lot.
* **Not perfect.** Some wrong predictions (red) sit right next to true contacts, others between helices that do not
  touch, and about a third of the top pairs are wrong. A 75,000-parameter model trained for half a minute captured only
  part of the coupling information (Section 4: almost 60 % of the context gain), and contact detection inherits that.
* **Why it works at all here.** In our family every coupling matrix is a mixture of three shared chemistry matrices.
  One attention head can then represent one "interaction type": its attention weights say *who* the partners are and
  its value/output matrices encode *which residues fit together*. This "attention ≈ factored Potts model" view is a
  useful mental model for protein language models (Bhattacharya *et al.* 2022 made it precise). In the toolkit's
  `simulate_protein_family`, by contrast, each contact gets its own *random* $20\times20$ table — no shared chemistry —
  and a model this small cannot store 128 unrelated tables; [Module 37](37_protein_sequences_language_models.ipynb)
  shows what happens then.

⚠️ **Pitfall.** Large attention between two positions is a *hint*, not a measurement. Here we could check it against
the true structure; on a real protein, treat attention- or Jacobian-derived contacts as predictions with error bars
and validate them (for example against a solved or predicted structure, [Module 38](38_protein_structure.ipynb)).

✏️ **Your turn.** Compute `precision_at_k(attn_scores, contacts, k)` for `k = L // 5` and for `k = L`. Why can the
precision at $k=L=68$ never exceed $38/68\approx0.56$ for this protein?

---
## 6 · 🧪 What did it learn? Function — zero-shot variant effects

> 📍 **Where we are.** If the model assigns probabilities to residues in context, it can score a mutation: a
> substitution to a residue that the model finds unlikely in this context is probably harmful. This is the idea of
> ESM-1v (Meier *et al.* 2021), and it requires no measurements at all — hence **zero-shot**.

🔬 **Assay box — deep mutational scanning (DMS).** *What is measured:* the fitness of thousands of variants of one
protein at once. A library containing (nearly) every single amino-acid substitution is built, coupled to a selection
(growth, binding, fluorescence-activated sorting), and the frequency of every variant before and after selection is
counted by sequencing; enrichment ratios become fitness scores (Fowler & Fields 2014). *Typical sizes:* $L\times19$ single
mutants for proteins of 50–1,000 residues. *Typical artefacts:* a non-linear readout (fitness saturates for stable
variants and floors for dead ones), noise from finite counts, and assay-specific selection pressures. The toolkit's
`deep_mutational_scan` mimics this: the true effect is $\Delta E$, the change in Potts score, and the measured fitness is
a sigmoid of $\Delta E$ plus noise.

**The masked-marginal score** (Meier *et al.* 2021): mask position $i$ in the wild type and compare the model's
log-probabilities of the mutant and the wild-type residue,

$$s(i,\,a_{\text{wt}}\!\to a)=\log p_\theta\big(x_i=a\mid x^{\text{wt}}_{-i}\big)-\log p_\theta\big(x_i=a_{\text{wt}}\mid x^{\text{wt}}_{-i}\big).$$

*In words:* how much less (or more) plausible the mutant residue is than the wild-type residue, *given the rest of this
particular protein*. The independent-site score is the same expression with the column frequencies, which ignore the
context.

In [ ]:
# ── 6.1 A simulated deep mutational scan and two zero-shot scores ──
rng = np.random.default_rng(SEED + 6)
dms = deep_mutational_scan(rng, wild_type, h_true, J_true)     # every single mutant of the wild type


@torch.no_grad()
def masked_marginals(model, seq):
    """log p(a | wild type with position i masked) for every position i and residue a (L × 20)."""
    model.eval()
    x_in = torch.tensor(seq)[None].repeat(L, 1)
    x_in[torch.arange(L), torch.arange(L)] = MASK                 # row i has position i masked
    logits, _, _ = model(x_in)
    return torch.log_softmax(logits[torch.arange(L), torch.arange(L)], dim=-1).numpy()


lp_wt = masked_marginals(model, wild_type)
pos, mut = dms["position"].to_numpy(), dms["mut"].map(AMINO_ACIDS.index).to_numpy()
wt_aa = wild_type[pos]
dms["score_transformer"] = lp_wt[pos, mut] - lp_wt[pos, wt_aa]
dms["score_independent"] = log_p_indep[pos, mut] - log_p_indep[pos, wt_aa]
print(dms.head(4).round(3).to_string(index=False))
rho = lambda a, b: spearmanr(a, b)[0]
print(f"\n{len(dms)} single mutants. Spearman correlation with ...      measured fitness   true ΔE")
for name, col in [("independent-site score", "score_independent"), ("transformer masked marginal", "score_transformer")]:
    print(f"  {name:32s}                       {rho(dms[col], dms.fitness):.3f}        {rho(dms[col], dms.delta_energy):.3f}")
print(f"  {'true ΔE itself (ceiling)':32s}                       {rho(dms.delta_energy, dms.fitness):.3f}        1.000")

🧮 **Math refresher — Spearman correlation.** Replace every value by its rank (1 = smallest) and compute the ordinary
Pearson correlation of the ranks. It equals 1 when one variable is *any* increasing function of the other, so it ignores
the non-linear readout of the assay and measures only whether the predictor *orders* the mutants correctly. That is why
DMS benchmarks such as ProteinGym report Spearman's $\rho$ (Module 37 explores this further).

In [ ]:
# ── 6.2 Zero-shot scores vs the measured fitness ──
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, col, name, c in zip(axes, ["score_independent", "score_transformer"],
                            ["independent-site model", "TinyEncoder masked marginal"], [OKABE_ITO[3], OKABE_ITO[0]]):
    ax.scatter(dms[col], dms.fitness, s=4, alpha=0.4, color=c)
    ax.set_xlabel(f"zero-shot score: {name}")
    ax.set_title(f"{name}\nSpearman ρ = {rho(dms[col], dms.fitness):.2f}")
axes[0].set_ylabel("measured fitness (SIMULATED DMS)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Both scores correlate with the measured fitness — conservation alone already tells you something
about which substitutions are tolerated ($\rho\approx0.49$) — but the transformer's context-aware score is much better
($\rho\approx0.78$), both against the noisy measurement and against the true effects $\Delta E$ (0.89 vs 0.59). The gain comes from the couplings: the effect of a
substitution depends on the residues at the contact partners *in this particular wild type*, which only a context
model can see. Notice also the ceiling: even the *true* $\Delta E$ does not correlate perfectly with the measured
fitness ($\rho\approx0.87$), because the assay saturates and is noisy; the transformer gets most of the way there. (On real
deep mutational scans the gain of context-aware models over site-independent ones is real but smaller, and absolute
correlations are lower: our simulated family is ideal and its couplings are strong.)

**A linear probe for secondary structure.** A common way to ask what an embedding "knows" is a **linear probe**:
freeze the model, take its per-residue embedding vectors, and fit a simple linear classifier to predict a property
(Rives *et al.* 2021 probed secondary structure this way). If a linear model can read the property from the
embeddings but not from the raw one-hot residues, the pre-training has made it explicit. Our property is helix vs loop.
Because every sequence in an aligned family shares the same structure, helix/loop is a property of the *position*; to
avoid simply memorising positions we train the probe on positions 0–35 and test it on the unseen positions 36–67.

In [ ]:
# ── 6.3 A linear probe: helix vs loop from embeddings vs one-hot residues (held-out positions) ──
with torch.no_grad():
    _, _, Z = model(X_test[:200])                           # per-residue embeddings, 200 × L × 64
Z = Z.numpy()
OH = np.eye(20)[X_test[:200].numpy()]                        # one-hot residues, 200 × L × 20
helix = np.array([s == "H" for s in ss], int)
train_pos, test_pos = np.arange(L) < 36, np.arange(L) >= 36
for name, feats in [("one-hot residue", OH), ("TinyEncoder embedding", Z)]:
    Xa, ya = feats[:, train_pos].reshape(-1, feats.shape[-1]), np.tile(helix[train_pos], len(feats))
    Xb, yb = feats[:, test_pos].reshape(-1, feats.shape[-1]), np.tile(helix[test_pos], len(feats))
    probe = LogisticRegression(C=0.01, class_weight="balanced", max_iter=2000).fit(Xa, ya)
    print(f"{name:22s}: balanced accuracy  train positions {balanced_accuracy_score(ya, probe.predict(Xa)):.2f}"
          f" | held-out positions {balanced_accuracy_score(yb, probe.predict(Xb)):.2f}   (chance = 0.50)")

🔍 **What just happened?** An honest, modest result. On the training positions the embedding probe is nearly perfect —
but that is memorisation of positions, which the learned position embeddings make easy. On the held-out positions the
embedding probe does only somewhat better than the one-hot probe, and both are far from perfect. Why so weak, when
real protein language models encode secondary structure well? (1) Our simulator puts no explicit helix signal in the
sequences (real helices are enriched in residues such as Ala, Leu and Glu and show a 3.6-residue hydrophobic periodicity);
helix and loop differ here only through their contact patterns. (2) There is a single family with a single fold, so the
probe sees just 36 training positions — real probes are trained across thousands of proteins. A probe can only reveal
what the data made learnable, which is a useful lesson about reading probing papers.

✏️ **Your turn.** Probe a different property: whether a position has at least two long-range contacts
(`(long_range.sum(1) >= 2)`). Does the embedding beat the one-hot residue on held-out positions? Then try a
**supervised** variant-effect model: fit a ridge regression ([Module 07](07_ridge_lasso_sparse_regression.ipynb)) from
`[score_transformer, score_independent]` to `fitness` on 100 random mutants and test on the rest.

### 📄 Research connection — Protein language models learn structure and function

- **What the field does.** Rives *et al.* (2021) trained transformers of up to 650 million parameters (ESM-1b) by masked-residue
    prediction on large sequence databases and showed with linear probes that their representations encode secondary
    structure, contacts and remote homology. Rao *et al.* (2021) showed that a sparse logistic regression on attention
    maps predicts contacts without any alignment, rivalling a classical coevolution method (Gremlin). Meier *et al.* (2021) introduced ESM-1v and the masked-marginal score for
    zero-shot variant-effect prediction, competitive with alignment-based models on deep mutational scans. Lin *et al.*
    (2023) scaled the approach to ESM-2 (up to 15 billion parameters) and built ESMFold, a structure predictor that
    needs a single sequence instead of an alignment. ProtTrans (Elnaggar *et al.* 2022) trained a range of such models
    in parallel.
- **What this notebook builds.** The same pipeline at toy scale with ground truth: masked-LM pre-training on one simulated family,
    attention-map contact extraction with symmetrisation and APC (precision@L/2 against the true map), a categorical
    Jacobian, masked-marginal zero-shot scores against a simulated DMS and against the true $\Delta E$, and a linear
    probe with held-out positions.
- **Where the frontier is.** Open questions (2023–2026): how much protein language models learn *physics* versus memorising
    family statistics (evidence that they largely store coevolution-like statistics of interacting motifs, Zhang *et
    al.* 2024); why scaling improves some tasks but not variant-effect prediction uniformly (larger is not always better
    on ProteinGym); how to calibrate scores for clinical use; and how to combine them with alignments and structures.
- **Read next.** Rives *et al.* 2021 (PNAS); Rao *et al.* 2021, *Transformer protein language models are unsupervised
    structure learners* (ICLR); Meier *et al.* 2021 (NeurIPS).

---
## 7 · Contrastive learning in one page

> 📍 **Where we are.** Masked modelling is one way for data to supervise itself. The other big family of
> self-supervised methods is **contrastive learning**, used for images (SimCLR, Chen *et al.* 2020), for pairing images
> with text, and in biology for matching molecules, cells or sequences across views.

**The idea.** Make two corrupted "views" of each example (two random maskings, crops, noise draws). Train an encoder
so that the two views of the *same* example have similar embeddings and views of *different* examples do not. For a
batch of $B$ examples with normalised embeddings $\mathbf z_b,\mathbf z'_b$ and cosine similarity
$\operatorname{sim}(\mathbf u,\mathbf v)=\mathbf u\cdot\mathbf v/(\lVert\mathbf u\rVert\lVert\mathbf v\rVert)$, the
**InfoNCE** loss is

$$\mathcal L_{\text{InfoNCE}}=-\frac1B\sum_{b=1}^{B}\log\frac{\exp\big(\operatorname{sim}(\mathbf z_b,\mathbf z'_b)/\tau\big)}
{\sum_{c=1}^{B}\exp\big(\operatorname{sim}(\mathbf z_b,\mathbf z'_c)/\tau\big)}.$$

*In words:* for each example, the softmax asks "which of the $B$ second views is my partner?" — a $B$-way
classification whose correct answer is the matching view; the loss is its cross-entropy, and $\tau$ is a temperature
(exactly the scaling of Section 2). Random guessing gives $\log B$. The other examples in the batch serve as
**negatives**, so no labels are needed.

In [ ]:
# ── 7.1 InfoNCE for two random maskings of the same sequences ──
def info_nce(z1, z2, tau=0.1):
    """Row b of z1 must pick out row b of z2 among all B rows (cross-entropy over cosine similarities / tau)."""
    z1, z2 = F.normalize(z1, dim=-1), F.normalize(z2, dim=-1)
    sim = z1 @ z2.T / tau
    return F.cross_entropy(sim, torch.arange(len(z1))).item(), sim.numpy()


B = 32
xb = X_test[:B]
view1, _ = mask_tokens(xb, gen=torch.Generator().manual_seed(SEED + 7))
view2, _ = mask_tokens(xb, gen=torch.Generator().manual_seed(SEED + 8))
torch.manual_seed(SEED + 9)
untrained = TinyEncoder(n_pos=L).eval()
with torch.no_grad():
    loss_trained, sim_trained = info_nce(model(view1)[2].mean(1), model(view2)[2].mean(1))   # mean-pooled embeddings
    loss_random, sim_random = info_nce(untrained(view1)[2].mean(1), untrained(view2)[2].mean(1))
print(f"InfoNCE with B = {B}: chance level log B = {np.log(B):.3f}")
print(f"  untrained encoder        : {loss_random:.3f}")
print(f"  masked-LM-trained encoder: {loss_trained:.3f}")
fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.6))
for ax, S, name in zip(axes, [sim_random, sim_trained], ["untrained encoder", "masked-LM encoder"]):
    im = ax.imshow(S, cmap="magma"); fig.colorbar(im, ax=ax, shrink=0.8)
    ax.set_xlabel("view 2 of sequence c"); ax.set_ylabel("view 1 of sequence b"); ax.set_title(f"similarity / τ: {name}")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Both losses are below chance ($\log 32=3.47$): even an untrained encoder maps two maskings of
the same sequence to somewhat similar vectors, because each view leaves 85 % of the positions untouched, so about
0.85² ≈ 72 % are untouched in both. The masked-LM encoder does
better (2.08 vs 2.62) although it was never trained for this task, but its similarity matrix shows large bright blocks:
groups of family members whose mean-pooled embeddings are alike, because masked pre-training rewards representing what
the members have in common, not what distinguishes one from another. Those members remain hard negatives. Making
sequence-level embeddings discriminative is exactly what contrastive training optimises: it would raise the diagonal
and darken the rest.

| | Masked modelling | Contrastive learning |
|---|---|---|
| Question asked | "what is hidden here?" | "which other view is mine?" |
| Output | per-token predictions (and embeddings) | one embedding per example |
| Needs | nothing but sequences | a good choice of views (augmentations) and many negatives |
| Natural uses | per-residue tasks, variant scoring, generation | retrieval, clustering, matching across modalities |
| Pitfall | learns whatever predicts tokens, including dataset quirks | learns whatever the augmentations leave unchanged |

✏️ **Your turn.** Change `tau` to 1.0 in `info_nce`. Why does the loss of both encoders move towards $\log B$? (Hint:
Section 2.2 — a large temperature flattens the softmax.)

---
## 8 · Scaling and foundation models

> 📍 **Where we are.** Our model has 75,000 parameters and saw one family. A **foundation model** is pre-trained once,
> at large scale, on broad data, and then reused for many tasks. Three practical issues decide what is possible:
> compute, tokenisation and the way the model is used downstream.

**Scale.** For language models, test loss falls smoothly (roughly as a power law) as parameters, data and compute grow
together (Kaplan *et al.* 2020, a preprint). Protein models show similar trends for the masked-LM loss and for contact
prediction (Lin *et al.* 2023), but downstream benefits are uneven: a bigger model is not automatically a better
variant-effect predictor.

**The $O(L^2)$ problem.** Every head of every layer computes an $L\times L$ matrix of attention weights. Memory and time
therefore grow with the *square* of the sequence length — fine for a protein domain, a serious obstacle for a
chromosome.

In [ ]:
# ── 8.1 The memory of one attention matrix grows like L² ──
def human_bytes(n):
    """Format a number of bytes with a readable unit."""
    for unit, f in [("PB", 1e15), ("TB", 1e12), ("GB", 1e9), ("MB", 1e6), ("kB", 1e3)]:
        if n >= f:
            return f"{n / f:,.1f} {unit}"
    return f"{n:.0f} B"


print("     L (tokens) | one L × L attention matrix (float32) | × 16 heads × 24 layers")
for n_tok in [68, 1_000, 10_000, 100_000, 1_000_000]:
    one = 4 * n_tok ** 2                                    # 4 bytes per float32 number
    print(f"  {n_tok:13,d} | {human_bytes(one):>20s}                 | {human_bytes(one * 16 * 24):>12s}")
print(f"\nTinyEncoder: {sum(p.numel() for p in model.parameters()):,} parameters; ESM-2 models: 8 million to 15 billion")

🔍 **What just happened?** A protein domain (hundreds of residues) needs kilobytes to megabytes per attention matrix;
a 100-kb genomic window would need 40 GB for a *single* matrix, before multiplying by heads and layers. (Optimised
implementations avoid storing the full matrix, but the computation still grows like $L^2$.) This is why genomic models
use long tokens, local or sparse attention, or non-attention architectures (state-space and long-convolution models
such as Hyena, used by HyenaDNA and Evo) — an active, fast-moving area in which claims should be read with care.

**Tokenisation.** What counts as one token is a design choice with large consequences. Proteins are almost always
tokenised by amino acid. For DNA the options include single nucleotides, overlapping $k$-mers (DNABERT), non-overlapping
$k$-mers (the Nucleotide Transformer) and **byte-pair encoding** (BPE, used by DNABERT-2), which repeatedly merges the most
frequent adjacent pair of tokens into a new token, so that common substrings become single tokens.

In [ ]:
# ── 8.2 Four ways to tokenise 1 kb of (SIMULATED) DNA ──
rng = np.random.default_rng(SEED + 10)
repeat = "".join(rng.choice(list("ACGT"), 12))                    # a 12-bp element that recurs (like a short repeat)
pieces = [repeat if rng.random() < 0.3 else "".join(rng.choice(list("ACGT"), 12)) for _ in range(84)]
dna = "".join(pieces)[:1000]


def bpe_tokenise(seq, n_merges):
    """Byte-pair encoding: start from single letters; repeatedly merge the most frequent adjacent pair."""
    toks, vocab = list(seq), set(seq)
    for _ in range(n_merges):
        (a, b), _ = Counter(zip(toks, toks[1:])).most_common(1)[0]
        merged, i = [], 0
        while i < len(toks):
            if i + 1 < len(toks) and toks[i] == a and toks[i + 1] == b:
                merged.append(a + b); i += 2
            else:
                merged.append(toks[i]); i += 1
        toks = merged; vocab.add(a + b)
    return toks, vocab


bpe_toks, bpe_vocab = bpe_tokenise(dna, 60)
rows = [("single nucleotides", 4, len(dna)),
        ("overlapping 6-mers (DNABERT)", 4 ** 6, len(dna) - 5),
        ("non-overlapping 6-mers (Nucleotide Transformer)", 4 ** 6, math.ceil(len(dna) / 6)),
        ("BPE after 60 merges (this toy)", len(bpe_vocab), len(bpe_toks))]
print(pd.DataFrame(rows, columns=["tokeniser", "vocabulary size", "tokens for 1,000 bp"]).to_string(index=False))
print("\nlongest BPE tokens:", sorted(bpe_vocab, key=len)[-3:], "| planted repeat:", repeat)

🔍 **What just happened?** Single nucleotides give the longest sequences (1,000 tokens) with the smallest vocabulary.
Overlapping $k$-mers do not shorten the sequence at all, and neighbouring tokens share 5 of 6 letters, which makes
masked prediction partly trivial. Non-overlapping 6-mers cut the length six-fold at the price of 4,096 vocabulary
entries and a dependence on the reading frame. BPE adapts to the data: after 60 merges its longest tokens are copies and pieces of
the planted repeat, because frequent substrings get their own tokens — just as BPE on English text learns whole words.
Since attention costs $L^2$, a 6-fold shorter sequence is 36 times cheaper.

**Three ways to use a pre-trained model.**

| Use | What is trained | Labels needed | Example in this notebook |
|---|---|---|---|
| **Zero-shot** | nothing | none | masked-marginal variant scores (Section 6) |
| **Probing** (frozen features) | a small head on top of fixed embeddings | a few hundred to thousands | linear probe for helix/loop (Section 6) |
| **Fine-tuning** | all (or some) weights, starting from the pre-trained ones | more; risk of over-fitting | — (✏️ below) |

Parameter-efficient fine-tuning (e.g. adding small low-rank "adapter" matrices and training only those) sits between
probing and full fine-tuning.

✏️ **Your turn.** Fine-tune: copy the model (`import copy; ft = copy.deepcopy(model)`), add a linear head that maps the
mean embedding to a number, and train it for 100 steps to predict the DMS fitness of 200 random single mutants
(feed the mutant sequences, unmasked). Compare its test Spearman correlation with the zero-shot score.

### 📄 Research connection — Genomic and single-cell foundation models, and how to evaluate them

- **What the field does.** The masked-LM recipe has been transferred to DNA — DNABERT (Ji *et al.* 2021; overlapping $k$-mers), the
    Nucleotide Transformer (Dalla-Torre *et al.* 2025; trained on genomes of many species), and long-context models such
    as Evo (Nguyen *et al.* 2024) — and to single-cell transcriptomes, where a cell is turned into a "sentence" of genes:
    Geneformer (Theodoris *et al.* 2023) orders genes by expression rank, scGPT (Cui *et al.* 2024) bins expression
    values. These models are pre-trained on millions of cells or many genomes and then probed or fine-tuned.
- **What this notebook builds.** The ingredients these models share — tokenisation choices (Section 8.2), masked pre-training (Section 4),
    the three modes of use (zero-shot, probing, fine-tuning) — and the habit of comparing against simple baselines and
    an oracle, which our simulation makes possible.
- **Where the frontier is.** Evaluation is the open problem. Kedzierska *et al.* (2025) found that, used zero-shot, Geneformer and
    scGPT embeddings were often outperformed by simple baselines such as highly variable genes or scVI on tasks like cell
    type clustering and batch integration. Genes in a cell have no natural order, so how to tokenise a transcriptome is
    still debated. For genomic models, held-out chromosomes do not remove evolutionary similarity between training and
    test sequences. A fair evaluation states what was seen in pre-training, uses strong simple baselines, and splits data
    to match the claim ([Module 42](42_trustworthy_ml.ipynb), [Module 43](43_llms_ai_agents.ipynb)).
- **Read next.** Ji *et al.* 2021, DNABERT (Bioinformatics); Theodoris *et al.* 2023, Geneformer (Nature); Kedzierska *et
    al.* 2025, zero-shot evaluation of single-cell foundation models (Genome Biology).

---
## 9 · 🧭 From toy to real data

> 📍 **Where we are.** Everything above ran on one simulated family in about a minute. Here is what changes with real
> sequences — no downloads, just orientation.

**Data.** Protein language models are trained on clustered sequence databases: [UniProt](https://www.uniprot.org)
and its UniRef clusters (UniRef50/90 remove near-duplicates so that large families do not dominate), often supplemented
with metagenomic sequences. Families and alignments live in [InterPro/Pfam](https://www.ebi.ac.uk/interpro/). Real
sequences are **unaligned** and of different lengths, so models need padding masks and relative positional encodings,
and a single model covers all families at once.

**Models and tools.** ESM-2 and ESM-1v ([github.com/facebookresearch/esm](https://github.com/facebookresearch/esm)),
ProtTrans, and many others are distributed through the [Hugging Face](https://huggingface.co) model hub and its
`transformers` library — a tool to use on Colab, not a dependency of this course. Variant-effect benchmarks:
[ProteinGym](https://proteingym.org) (Notin *et al.* 2023) collects hundreds of deep mutational scans. Genomic models:
DNABERT, Nucleotide Transformer, Evo; single-cell: Geneformer, scGPT, typically applied to atlases such as
[CELLxGENE](https://cellxgene.cziscience.com).

**Compute.** Pre-training ESM-2-scale models takes large GPU clusters for weeks; *using* a small pre-trained model
(8–35 million parameters) to embed or score a few thousand sequences runs on a free Colab GPU or even a CPU. Fine-tuning
a model of a few hundred million parameters needs a single modern GPU.

**The first three things that go wrong.**

1. **Homology leakage.** Test proteins that are close relatives of training proteins inflate performance; split by
   sequence identity or by family, and remember that a public pre-trained model may already have seen your "test"
   sequences during pre-training.
2. **Weak baselines.** A context-free conservation score (Section 6's independent-site model) or an alignment-based
   Potts model ([Module 37](37_protein_sequences_language_models.ipynb)) is a strong baseline for variant effects; so
   are highly variable genes and PCA for single cells. Report them.
3. **Mismatched objectives.** A language model scores how *evolutionarily plausible* a sequence is. That is related to,
   but not the same as, the property your assay measures (activity at 37 °C, binding to a new target, expression in
   yeast); correlations vary widely from assay to assay, and scores need calibration before any clinical use.

✏️ **Your turn.** Pick a protein you care about and write down, before touching any model: how many homologous
sequences UniRef90 contains for it (relative to its length), whether a deep mutational scan exists in ProteinGym or
MaveDB, which held-out split would test the claim you want to make, and which of the three failure modes above is most
likely to bite.

## 📌 Key takeaways

- **Context matters in proteins:** residues in 3-D contact co-vary, so the best residue at one position depends on its
  partners; models that read the whole sequence can exploit this, profile/independent-site models cannot.
- **Attention** $=\operatorname{softmax}(QK^\top/\sqrt{d_k})V$: every position takes a weighted average of values, with
  weights from query–key similarity. The $\sqrt{d_k}$ keeps the softmax from saturating; with suitable features
  attention *is* a kernel smoother — a soft, learned k-NN.
- Self-attention is **permutation-equivariant**; **positional encodings** (sinusoidal, learned, rotary) supply order.
- A **transformer block** = multi-head attention + residual + LayerNorm + position-wise MLP; parameters do not depend on
  sequence length (except learned positions), but attention costs $O(L^2)$.
- **Masked language modelling** turns unlabelled sequences into a supervised task. On our family a 75k-parameter model
  beat the independent-site model and captured almost 60 % of the context information available to the true model.
- **What it learned, checked against truth:** attention maps and the categorical Jacobian recovered most long-range
  contacts (precision@L/2 ≈ 0.6–0.65 vs 0.02 by chance), and masked-marginal scores predicted mutation effects better
  than conservation; a linear probe for helix/loop was only weakly better than one-hot — probes reveal only what the
  data made learnable.
- **Contrastive learning** (InfoNCE) asks "which view is mine?" instead of "what is hidden?".
- **Foundation models** for proteins, DNA and cells reuse this recipe at scale; judge them by honest splits, strong
  simple baselines and the match between the pre-training objective and the biological question.

---
## ✅ Checkpoint

1. A query $\mathbf q=(2,0,2,0)$ attends to keys $\mathbf k_1=(1,0,1,0)$, $\mathbf k_2=(0,1,0,1)$, $\mathbf k_3=(1,1,0,0)$ with values $v_1=1, v_2=0, v_3=1$. Compute the attention weights and the output.
   <details><summary>Answer</summary>Dot products 4, 0, 2; divided by $\sqrt4=2$: 2, 0, 1; softmax $\propto e^2, e^0, e^1$ = 7.39, 1, 2.72 → weights 0.665, 0.090, 0.245. Output $=0.665\cdot1+0.090\cdot0+0.245\cdot1=0.910$.</details>
2. Why is $QK^\top$ divided by $\sqrt{d_k}$?
   <details><summary>Answer</summary>For independent unit-variance entries, $\mathbf q\cdot\mathbf k$ has variance $d_k$. Without scaling, long keys give huge scores, the softmax saturates on one key and its gradients vanish; dividing by $\sqrt{d_k}$ restores unit variance.</details>
3. What happens to a transformer for proteins if you remove the positional encodings?
   <details><summary>Answer</summary>Self-attention becomes permutation-equivariant: the model sees a *bag* of residues. It can no longer learn that position 12 should read position 45, so it cannot learn contact-specific co-variation (or anything order-dependent).</details>
4. Why does masked language modelling on a protein family learn co-variation, and hence contacts?
   <details><summary>Answer</summary>To predict a hidden residue better than its column frequencies, the model must use residues that carry information about it — in a family these are mainly its 3-D contact partners, because contacting residues co-vary. Attending to them lowers the loss, so the attention pattern comes to reflect the contact map.</details>
5. What is the perplexity of a uniform model over 20 amino acids? Of a model that always gives the true residue probability 0.5?
   <details><summary>Answer</summary>Uniform: $e^{-\log(1/20)}=20$. Probability 0.5: $e^{\log 2}=2$ — it is as uncertain as a fair choice between 2 residues.</details>
6. How many parameters does one multi-head attention layer with $d=64$ have (any number of heads), including biases? Why does the number not depend on the sequence length?
   <details><summary>Answer</summary>$W_{QKV}$: $64\cdot192+192$; $W_O$: $64\cdot64+64$; total $4\cdot64^2+4\cdot64=16{,}640$. The same matrices are applied at every position (weight sharing); only the $L\times L$ attention *weights* grow with length, and they are computed, not learned.</details>
7. Code reading: in `mask_tokens`, what fraction of all positions are replaced by `[MASK]`, and why are 10 % of the chosen positions left unchanged?
   <details><summary>Answer</summary>$0.15\times0.8=12\%$ become `[MASK]` (1.5 % become a random residue, 1.5 % stay unchanged but are still predicted). Leaving some unchanged (and randomising some) stops the model from learning that only `[MASK]` positions need a prediction, since at use time inputs contain no `[MASK]`.</details>
8. Write the masked-marginal score of mutation $a_{wt}\to a$ at position $i$. Why is it called zero-shot?
   <details><summary>Answer</summary>$\log p(x_i=a\mid x^{wt}_{-i})-\log p(x_i=a_{wt}\mid x^{wt}_{-i})$ with position $i$ masked. It uses only the pre-trained model — no fitness measurements of this protein are used for training — hence 'zero' labelled examples.</details>

## 🧠 Questions & discussion

1. Our attention maps recovered contacts well because the simulated couplings followed three shared chemical rules. What would you expect for a family whose couplings are arbitrary (every contact its own random table)? What does this suggest about *why* large protein language models, trained across many families, learn contacts?
2. A protein language model gives a missense variant a very low masked-marginal score. List three reasons why this variant might nevertheless be harmless in a patient, and one reason why a high score does not prove it is benign.
3. Attention weights are often visualised as 'what the model looks at'. In which situations is this interpretation misleading? How did Section 5 avoid relying on it?
4. Sequence databases over-represent some organisms (human, mouse, model microbes, pathogens) and some protein families. How could this bias zero-shot predictions for proteins from under-studied organisms, and how would you test it?
5. Masked and contrastive pre-training ask different questions. For (a) predicting the effect of point mutations and (b) retrieving functionally similar enzymes from a database, which would you choose and why?

## 🚀 Bonus — going further

- **Causal (GPT-style) vs masked training.** Replace masking by next-residue prediction: add a causal mask (scores set to $-\infty$ for $j>i$) to `MultiHeadSelfAttention` and train on the same family. Compare the single-site perplexity and the contact precision. Module 43 discusses autoregressive language models.
- **Fewer sequences.** Retrain on 250 and on 1,000 training sequences and plot contact precision@L/2 against family size. At what depth do the contacts disappear? (Coevolution methods show the same dependence, Module 37.)
- **Random couplings.** Generate a family with the toolkit's `simulate_protein_family(rng, contacts, n_seqs=3000)` (each contact gets a random 20 × 20 table) and train the same `TinyEncoder`. Does it still beat the independent-site model? Relate the answer to the number of parameters needed to store 128 unrelated tables.
- **The MSA Transformer idea.** Instead of one sequence, feed a small alignment and alternate attention along rows (positions) and columns (sequences), as in Rao *et al.* (2021, ICML). Sketch the tensor shapes for 16 sequences × 68 positions.
- **Real models on Colab.** With the Hugging Face `transformers` library, load the smallest ESM-2 checkpoint (8 million parameters), compute masked-marginal scores for a protein you care about, and compare them with a ProteinGym assay of that protein.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 26 · Attention, transformers and self-supervised learning
Biological question: can a model trained only to fill in masked residues discover contacts and mutation effects?
Data (simulated): simulate_helix_bundle (68 residues, contact map) + a Potts family sampled in Section 1
                  (3,000 sequences; couplings = mixtures of 3 chemistry matrices); deep_mutational_scan of the
                  wild type. Ground truth: contacts, h, J, ΔE of every single mutant.
Method: NumPy attention; PyTorch TinyEncoder (d = 64, 4 heads, 2 layers) pre-trained by BERT-style masking
Key equation: Attention(Q, K, V) = softmax(Q Kᵀ / √d_k) V ;  L = −(1/|M|) Σ_{i∈M} log p(x_i | x̃)
Key parameters: d_model, heads, layers, mask rate, steps, learning rate, family size
Ground-truth check result: perplexity (uniform / independent / transformer / oracle) = … ;
                           contact precision@L/2 (attention, Jacobian) = … ; Spearman vs DMS = …
What changed when I modified … (mask rate / family size / τ / d_model): …
One failure mode: (e.g. probe memorises positions; random-coupling family not learnable by a tiny model)
Connection to research: Rives 2021 / Rao 2021 / Meier 2021 / Lin 2023 (ESM); tool: ESM-2 via Hugging Face
```

## 📚 References

- Bahdanau D., Cho K., Bengio Y. (2015) Neural machine translation by jointly learning to align and translate. *ICLR*.
- Vaswani A. *et al.* (2017) Attention is all you need. *NeurIPS*.
- Devlin J., Chang M.-W., Lee K., Toutanova K. (2019) BERT: pre-training of deep bidirectional transformers for language understanding. *NAACL*.
- Chen T., Kornblith S., Norouzi M., Hinton G. (2020) A simple framework for contrastive learning of visual representations (SimCLR). *ICML*.
- Kaplan J. *et al.* (2020) Scaling laws for neural language models. arXiv:2001.08361 (preprint).
- Rives A. *et al.* (2021) Biological structure and function emerge from scaling unsupervised learning to 250 million protein sequences. *PNAS* 118:e2016239118.
- Rao R., Meier J., Sercu T., Ovchinnikov S., Rives A. (2021) Transformer protein language models are unsupervised structure learners. *ICLR*.
- Rao R. M. *et al.* (2021) MSA Transformer. *ICML*.
- Meier J. *et al.* (2021) Language models enable zero-shot prediction of the effects of mutations on protein function. *NeurIPS*.
- Vig J. *et al.* (2021) BERTology meets biology: interpreting attention in protein language models. *ICLR*.
- Bhattacharya N. *et al.* (2022) Interpreting Potts and transformer protein models through the lens of simplified attention. *Pacific Symposium on Biocomputing* 27.
- Zhou Z. *et al.* (2024) DNABERT-2: efficient foundation model and benchmark for multi-species genome. *ICLR*.
- Nguyen E. *et al.* (2023) HyenaDNA: long-range genomic sequence modeling at single nucleotide resolution. *NeurIPS*.
- Zhang Z. *et al.* (2024) Protein language models learn evolutionary statistics of interacting sequence motifs. *PNAS* 121.
- Lin Z. *et al.* (2023) Evolutionary-scale prediction of atomic-level protein structure with a language model. *Science* 379:1123–1130.
- Elnaggar A. *et al.* (2022) ProtTrans: toward understanding the language of life through self-supervised learning. *IEEE TPAMI* 44:7112–7127.
- Notin P. *et al.* (2023) ProteinGym: large-scale benchmarks for protein fitness prediction and design. *NeurIPS Datasets & Benchmarks*.
- Fowler D. M., Fields S. (2014) Deep mutational scanning: a new style of protein science. *Nat Methods* 11:801–807.
- Dunn S. D., Wahl L. M., Gloor G. B. (2008) Mutual information without the influence of phylogeny or entropy dramatically improves residue contact prediction. *Bioinformatics* 24:333–340.
- Miyazawa S., Jernigan R. L. (1985) Estimation of effective interresidue contact energies from protein crystal structures: quasi-chemical approximation. *Macromolecules* 18:534–552.
- Kyte J., Doolittle R. F. (1982) A simple method for displaying the hydropathic character of a protein. *J Mol Biol* 157:105–132.
- Ji Y., Zhou Z., Liu H., Davuluri R. V. (2021) DNABERT: pre-trained bidirectional encoder representations from transformers model for DNA-language in genome. *Bioinformatics* 37:2112–2120.
- Dalla-Torre H. *et al.* (2025) Nucleotide Transformer: building and evaluating robust foundation models for human genomics. *Nat Methods*.
- Nguyen E. *et al.* (2024) Sequence modeling and design from molecular to genome scale with Evo. *Science*.
- Theodoris C. V. *et al.* (2023) Transfer learning enables predictions in network biology. *Nature* 618:616–624.
- Cui H. *et al.* (2024) scGPT: toward building a foundation model for single-cell multi-omics using generative AI. *Nat Methods* 21:1470–1480.
- Kedzierska K. Z., Crawford L., Amini A. P., Lu A. X. (2025) Zero-shot evaluation reveals limitations of single-cell foundation models. *Genome Biol*.
- Goodfellow I., Bengio Y., Courville A. (2016) *Deep Learning*. MIT Press.

## ➡️ Next

Attention let every token attend to every other token — in effect, a complete graph with learned edge weights.
Real structured data usually hands you the graph already: a molecule's bonds, a signalling network's edges, a
protein's contact map. **[Module 25 · Graph neural networks and geometric deep learning](25_graph_neural_networks.ipynb)**
restricts message passing to that known structure and adds the symmetries — permutation and rotation invariance —
a graph demands. For the full coevolution story behind this notebook's protein-family data (mutual information,
Potts models, DCA, epistasis), see **[Module 37](37_protein_sequences_language_models.ipynb)**.